<div style="
    background: linear-gradient(135deg, #2D1E2F 0%, #5A2A5E 45%, #2C6E8F 100%);
    padding: 34px;
    border-radius: 14px;
    margin: 10px 0 20px 0;
    box-shadow: 0 10px 26px rgba(45,30,47,0.34);
    border: 1px solid rgba(255,255,255,0.12);
">
    <h1 style="
        color: #ffffff;
        margin: 0;
        text-align: center;
        font-size: 34px;
        font-weight: 300;
        letter-spacing: 2px;
    ">PHY-3500 · Notebook 03</h1>
    <p style="
        color: rgba(255,255,255,0.92);
        margin: 10px 0 0 0;
        text-align: center;
        font-size: 18px;
        letter-spacing: 0.5px;
    ">Autoencodeur MLP pour réduction de dimension spectrale</p>
</div>

<div style="
    background: linear-gradient(135deg, rgba(90,42,94,0.14), rgba(44,110,143,0.12));
    border: 1px solid rgba(90,42,94,0.30);
    border-left: 6px solid #5A2A5E;
    border-radius: 10px;
    padding: 18px 20px;
    margin: 14px 0;
">
<strong>Équipe :</strong> Alex, Justine<br>
<strong>Cours :</strong> PHY-3500 Physique numérique - Université Laval<br>
<strong>Prérequis :</strong> Notebook 01 exécuté (features + PCAAnalyzer disponibles)
</div>

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.14), rgba(45,30,47,0.14));
    border: 1px solid rgba(44,110,143,0.30);
    border-radius: 10px;
    padding: 18px 20px;
    margin: 14px 0;
">
<h3 style="margin: 0 0 10px 0; color: #5A2A5E;">Objectifs</h3>
<ol style="margin: 0; padding-left: 20px; line-height: 1.6;">
  <li>Entraîner un autoencodeur MLP sur les features spectrales LAMOST DR5.</li>
  <li>Explorer l’espace latent 2D et le comparer à PCA et UMAP.</li>
  <li>Comparer les performances de reconstruction avec PCA.</li>
  <li>Tester la continuité de l’espace latent par interpolation.</li>
  <li>Produire une synthèse comparative des méthodes.</li>
</ol>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 10px;
    padding: 12px 14px;
    margin-top: 10px;
">
<strong>Positionnement scientifique :</strong> ce notebook évalue une réduction de dimension non linéaire paramétrique (autoencodeur) et la confronte à deux baselines non supervisées (PCA, UMAP).
</div>

<a id="toc"></a>

## Table des matières

<div style="display: grid; grid-template-columns: repeat(2, minmax(260px, 1fr)); gap: 12px; margin: 14px 0 22px 0;">

  <div style="background: linear-gradient(135deg, #5A2A5E, #2C6E8F); padding: 14px; border-radius: 10px;">
    <a href="#setup" style="color: white; text-decoration: none;"><strong>0. Imports et configuration</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #2C6E8F, #5A2A5E); padding: 14px; border-radius: 10px;">
    <a href="#data" style="color: white; text-decoration: none;"><strong>1. Chargement des données</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #5A2A5E, #3D405B); padding: 14px; border-radius: 10px;">
    <a href="#arch" style="color: white; text-decoration: none;"><strong>2. Architecture et entraînement</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #3D405B, #2C6E8F); padding: 14px; border-radius: 10px;">
    <a href="#latent" style="color: white; text-decoration: none;"><strong>3. Espace latent 2D</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #2C6E8F, #5A2A5E); padding: 14px; border-radius: 10px;">
    <a href="#recon" style="color: white; text-decoration: none;"><strong>4. Qualité de reconstruction</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #5A2A5E, #3D405B); padding: 14px; border-radius: 10px;">
    <a href="#interp" style="color: white; text-decoration: none;"><strong>5. Continuité latente</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #3D405B, #2C6E8F); padding: 14px; border-radius: 10px;">
    <a href="#synthesis" style="color: white; text-decoration: none;"><strong>6. Synthèse comparative</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #2C6E8F, #5A2A5E); padding: 14px; border-radius: 10px;">
    <a href="#save" style="color: white; text-decoration: none;"><strong>7. Sauvegarde des sorties</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #5A2A5E, #2D1E2F); padding: 14px; border-radius: 10px; grid-column: span 2;">
    <a href="#conclusion" style="color: white; text-decoration: none;"><strong>Conclusion et références</strong></a>
  </div>

</div>

<a id="setup"></a>

<div style="background: linear-gradient(135deg, #5A2A5E 0%, #2C6E8F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(90,42,94,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">0 · Imports et configuration</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
Cette cellule fixe l’environnement d’exécution, les chemins d’artefacts et la graine aléatoire. C’est la base de la reproductibilité expérimentale du notebook.
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# ── Environnement projet ───────────────────────────────────────────────────
sys.path.insert(0, str(Path("../..").resolve() / "src"))

from utils import setup_project_env, latest_file

paths = setup_project_env(verbose=True)

# ── Chemins ────────────────────────────────────────────────────────────────
FEATURES_PATH = latest_file(paths["PROCESSED_DIR"], "features_*.csv")
if FEATURES_PATH is None:
    raise FileNotFoundError(
        "Aucun fichier features_*.csv trouvé dans data/reports/\n"
        "→ Lancer d'abord 00_master_pipeline.ipynb pour générer les features."
    )
FEATURES_PATH  = Path(FEATURES_PATH)
CATALOG_PATH   = Path(paths["CATALOG_DIR"]) / "master_catalog_gaia.csv"
FIGURES_ROOT   = Path(paths["REPORTS_DIR"]) / "figures" / "phy3500"
FIGURES_DIR    = FIGURES_ROOT / FEATURES_PATH.stem
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR     = Path(paths["REPORTS_DIR"]) / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nFeatures  : {FEATURES_PATH}")
print(f"Catalog   : {CATALOG_PATH}")
print(f"Figures   : {FIGURES_DIR}")
print(f"Models    : {MODELS_DIR}")

# ── Imports dimred ─────────────────────────────────────────────────────────
from dimred import DimRedDataLoader, PCAAnalyzer, DimRedVisualizer
from dimred.autoencoder import SpectralAutoencoder

# ── Logging & reproductibilité ─────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("\n✓ Environnement prêt.")

<a id="data"></a>

<div style="background: linear-gradient(135deg, #2C6E8F 0%, #3D405B 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(44,110,143,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1 · Chargement des données</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
Le pipeline de chargement est identique aux notebooks 01 et 02 pour garantir une comparaison rigoureuse entre méthodes.
</div>

Les variables d’entrée sont standardisées avant l’apprentissage :

$$
x'_j = \frac{x_j - \mu_j}{\sigma_j}
$$

Cette étape est essentielle car l’autoencodeur optimise une perte MSE sensible aux échelles des features.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #3D405B; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
loader = DimRedDataLoader(
    features_path=FEATURES_PATH,
    catalog_path=CATALOG_PATH,
    random_state=RANDOM_STATE,
)

X, y, meta = loader.load(
    mode="features",
    spectro_only=True,
    snr_min=10.0,
    scale=True,          # StandardScaler — obligatoire pour l'AE
    class_balance=False,
    classes=["STAR", "GALAXY", "QSO"],
)

feature_names = loader.get_feature_names()
input_dim     = X.shape[1]

print(f"\nDonnées chargées :")
print(f"  X         : {X.shape}")
print(f"  y         : {y.shape} | classes = {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"  meta      : {meta.shape}")
print(f"  input_dim : {input_dim}")

In [ ]:
# Chargement du PCAAnalyzer sauvegardé (depuis notebook 01)
# ── Résilient à 3 cas : absent | incompatible | valide ────────────────────
pca_model_path = MODELS_DIR / "pca_analyzer.joblib"
_pca_loaded    = False

if pca_model_path.exists():
    try:
        pca = PCAAnalyzer.load(str(pca_model_path))

        # Vérification de compatibilité dimensionnelle
        n_expected = getattr(pca.sklearn_pca, "n_features_in_", None)
        if n_expected is not None and n_expected != X.shape[1]:
            raise ValueError(
                f"Dimensions incompatibles : modèle entraîné sur {n_expected} features, "
                f"X courant a {X.shape[1]} features (spectro_only a changé le nombre de features)."
            )

        scores_pca = pca.transform(X)
        _pca_loaded = True
        print(f"✓ PCAAnalyzer chargé ({pca.sklearn_pca.n_components_} composantes | "
              f"{X.shape[1]} features)")
        print(f"  scores_pca : {scores_pca.shape}")

    except Exception as e:
        print(f"⚠  PCAAnalyzer incompatible → réentraînement")
        print(f"   Raison : {e}")

if not _pca_loaded:
    print("▲  Entraînement PCAAnalyzer sur X courant...")
    pca = PCAAnalyzer(n_components=50, random_state=RANDOM_STATE)
    scores_pca = pca.fit_transform(X, feature_names=feature_names)
    pca.save(str(pca_model_path))
    print(f"✓  PCAAnalyzer entraîné et sauvegardé ({X.shape[1]} features → {scores_pca.shape[1]} PCs)")

n_pcs_95 = pca.n_components_for_variance(0.95)
print(f"  95% variance → {n_pcs_95} composantes PCA")


<a id="arch"></a>

<div style="background: linear-gradient(135deg, #3D405B 0%, #5A2A5E 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(61,64,91,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">2 · Architecture et entraînement de l’autoencodeur</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(90,42,94,0.10), rgba(44,110,143,0.10));
    border-left: 6px solid #5A2A5E;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<strong>Architecture choisie :</strong> X(D) -> 256 -> 128 -> 64 -> z(2) -> 64 -> 128 -> 256 -> Xhat(D)<br>
<strong>Activation :</strong> ReLU + BatchNorm + Dropout(0.1)<br>
<strong>Loss :</strong> MSE<br>
<strong>Optimisation :</strong> Adam + ReduceLROnPlateau + early stopping
</div>

Objectif d’apprentissage (forme simplifiée) :

$$
\mathcal{L}(\theta)=\frac{1}{N}\sum_{i=1}^N\|x_i-\hat x_i\|_2^2 + \lambda\|\theta\|_2^2
$$

où :
- le premier terme force la fidélité de reconstruction ;
- le second (weight decay) régularise les paramètres pour limiter le surapprentissage.

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin-bottom: 10px;
">
Le choix de z=2 est volontaire : il impose une compression sévère pour comparer directement Autoencodeur, PCA (PC1/PC2) et UMAP sur un même plan 2D.
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Construction du modèle ────────────────────────────────────────────────
ae = SpectralAutoencoder(
    input_dim=input_dim,
    latent_dim=2,               # 2D pour comparaison directe PCA / UMAP
    hidden_dims=[256, 128, 64],
    dropout=0.1,
)

print(ae)
print(f"\nDevice : {ae.device}")

# Compte des paramètres (info pour le rapport)
try:
    import torch
    n_params = sum(p.numel() for p in ae._model.parameters()
                   if p is not None and p.requires_grad)
    # Reconstruit _model pour le comptage avant fit()
except Exception:
    pass

In [ ]:
# ── Entraînement ─────────────────────────────────────────────────────────
# Vérifie si un modèle sauvegardé existe déjà
ae_model_path = MODELS_DIR / "ae_latent2.pt"

if ae_model_path.exists():
    print("✓ Modèle autoencodeur trouvé — chargement...")
    ae = SpectralAutoencoder.load(str(ae_model_path))
    history = ae.history_
    print(f"  fit_time : {ae.fit_time_:.1f}s")
    print(f"  best val_loss : {min(history['val_loss']):.6f}")
else:
    print("Entraînement de l'autoencodeur (latent_dim=2)...")
    history = ae.fit(
        X,
        epochs=200,
        lr=1e-3,
        batch_size=512,
        val_fraction=0.10,
        weight_decay=1e-5,
        lr_scheduler=True,
        early_stopping_patience=25,
        verbose=True,
    )
    ae.save(str(ae_model_path))
    print(f"\n✓ Modèle sauvegardé : {ae_model_path}")

print(f"\nEpochs effectuées : {len(history['train_loss'])}")
print(f"Best val_loss     : {min(history['val_loss']):.6f}")

<a id="ae3d"></a>

<div style="background: linear-gradient(135deg, #3D405B 0%, #5A2A5E 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(61,64,91,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">2.bis · Autoencodeur z=3 — un axe latent de plus</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(90,42,94,0.10), rgba(44,110,143,0.10));
    border-left: 6px solid #5A2A5E;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<p style="margin: 0 0 8px 0;">Le modèle z=2 compresse vers un plan bidimensionnel. On s'interroge ici : <strong>un troisième axe latent encode-t-il un paramètre physique indépendant</strong> (métallicité, gravité de surface) que les deux premiers n'ont pas capté ?</p>
<p style="margin: 0;">Cette question est directement comparable au UMAP 3D de NB02, où l'axe 3 corrélait faiblement avec log g (ρ=−0.21). L'AE, étant paramétrique et optimisé pour la reconstruction, pourrait trouver un troisième axe plus informatif.</p>
</div>

$$
f_\theta: \mathbb{R}^{183} \rightarrow \mathbb{R}^{3}, \quad z=f_\theta(x)
\qquad g_\phi: \mathbb{R}^{3} \rightarrow \mathbb{R}^{183}, \quad \hat x=g_\phi(z)
$$

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Autoencodeur z=3 — entraînement ou chargement ────────────────────────
ae3_model_path = MODELS_DIR / "ae_latent3.pt"

# Guard de cohérence dimensionnelle
_ae3_loaded = False
if ae3_model_path.exists():
    _tmp = SpectralAutoencoder.load(str(ae3_model_path))
    saved_dim3 = getattr(_tmp._model.encoder.net[0], 'in_features', None)
    if saved_dim3 == input_dim:
        ae3 = _tmp
        history3 = ae3.history_
        _ae3_loaded = True
        print(f"✓ Checkpoint ae_latent3.pt chargé | input_dim={saved_dim3} | "
              f"epochs={len(history3['train_loss'])} | "
              f"best_val={min(history3['val_loss']):.5f}")
    else:
        print(f"⚠  Checkpoint périmé (saved={saved_dim3}, current={input_dim}) → réentraînement.")
        ae3_model_path.unlink()

if not _ae3_loaded:
    print("Entraînement AE z=3 (latent_dim=3)...")
    ae3 = SpectralAutoencoder(
        input_dim=input_dim,
        latent_dim=3,
        hidden_dims=[256, 128, 64],
        dropout=0.1,
    )
    history3 = ae3.fit(
        X,
        epochs=200,
        lr=1e-3,
        batch_size=512,
        val_fraction=0.10,
        weight_decay=1e-5,
        lr_scheduler=True,
        early_stopping_patience=25,
        verbose=True,
    )
    ae3.save(str(ae3_model_path))
    print(f"\n✓ Modèle sauvegardé : {ae3_model_path}")

print(f"\nEpochs : {len(history3['train_loss'])} | Best val_loss : {min(history3['val_loss']):.6f}")

# ── Encodage ──────────────────────────────────────────────────────────────
Z_ae3 = ae3.encode(X)
print(f"Espace latent z=3 : {Z_ae3.shape}")

# ── Corrélations des 3 axes avec paramètres physiques ─────────────────────
from scipy.stats import spearmanr
import pandas as pd

phys_cols = [
    ("teff_gspphot",  "T_eff"),
    ("logg_gspphot",  "log g"),
    ("mh_gspphot",    "[Fe/H]"),
    ("bp_rp",         "BP-RP"),
]

print()
print("Corrélations Spearman — AE z=3 vs paramètres Gaia")
print(f"{'Paramètre':<12} {'Axe 1':>10} {'Axe 2':>10} {'Axe 3':>10}")
print("-" * 46)
corr3_results = {}
for col, label in phys_cols:
    if col not in meta.columns:
        continue
    vals = meta[col].values.astype(float)
    valid = np.isfinite(vals)
    r1, _ = spearmanr(Z_ae3[valid, 0], vals[valid])
    r2, _ = spearmanr(Z_ae3[valid, 1], vals[valid])
    r3, _ = spearmanr(Z_ae3[valid, 2], vals[valid])
    print(f"{label:<12} {r1:>+10.3f} {r2:>+10.3f} {r3:>+10.3f}")
    corr3_results[label] = (r1, r2, r3)
print("-" * 46)

# Axe 3 : quelle est sa meilleure corrélation ?
best3_param, best3_rho = max(
    ((label, abs(rs[2])) for label, rs in corr3_results.items()),
    key=lambda x: x[1]
)
print(f"\nAxe 3 encode principalement : {best3_param} (|ρ| = {best3_rho:.3f})")

# ── MSE reconstruction z=3 vs z=2 ────────────────────────────────────────
X_recon3 = ae3.reconstruct(X)
mse3 = float(np.mean((X - X_recon3) ** 2))
mse2_ref = float(np.mean((X - ae.reconstruct(X)) ** 2))  # ae est z=2

print()
print(f"MSE reconstruction z=2 : {mse2_ref:.5f}")
print(f"MSE reconstruction z=3 : {mse3:.5f}")
print(f"Gain z=2→z=3            : {(mse2_ref - mse3) / mse2_ref * 100:.1f}%")

# ── Figure : embedding 3D (Plotly interactif) ────────────────────────────
try:
    import plotly.express as px
    _PLOTLY_OK3 = True
except ImportError:
    print("⚠  Plotly non installé → figure statique uniquement.")
    _PLOTLY_OK3 = False

if _PLOTLY_OK3:
    df3d = pd.DataFrame({
        "z1": Z_ae3[:, 0], "z2": Z_ae3[:, 1], "z3": Z_ae3[:, 2],
        "T_eff": meta["teff_gspphot"].values,
        "[Fe/H]": meta["mh_gspphot"].values,
        "log g": meta["logg_gspphot"].values,
        "Classe": y,
    })
    N_PLT3 = min(20_000, len(df3d))
    rng3 = np.random.default_rng(42)
    df3d_s = df3d.iloc[rng3.choice(len(df3d), N_PLT3, replace=False)].copy()

    fig3d = px.scatter_3d(
        df3d_s.dropna(subset=["T_eff"]),
        x="z1", y="z2", z="z3",
        color="T_eff",
        color_continuous_scale="plasma",
        range_color=[
            float(df3d_s["T_eff"].quantile(0.02)),
            float(df3d_s["T_eff"].quantile(0.98)),
        ],
        opacity=0.6,
        title=f"AE z=3 — T_eff | axe 3 ↔ {best3_param} (|ρ|={best3_rho:.3f})",
    )
    fig3d.update_traces(marker=dict(size=2))
    fig3d.update_layout(
        scene=dict(
            xaxis_title="AE axe 1",
            yaxis_title="AE axe 2",
            zaxis_title=f"AE axe 3 (↔ {best3_param})",
        ),
        margin=dict(l=0, r=0, b=0, t=50),
    )
    path3d = FIGURES_DIR / "ae3d_teff.html"
    fig3d.write_html(str(path3d))
    print(f"✓ Figure 3D interactive sauvegardée : {path3d}")
    fig3d.show()

<div style="
    background: linear-gradient(135deg, rgba(90,42,94,0.10), rgba(44,110,143,0.10));
    border-left: 6px solid #3D405B;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #3D405B;">Interprétation — AE z=3 vs UMAP 3D</h4>
<p style="margin: 0 0 8px 0;">Si l'axe 3 de l'AE corrèle plus fortement avec un paramètre physique que l'axe 3 UMAP (|ρ|=0.21 avec log g dans NB02), cela démontre l'avantage de l'approche paramétrique : l'AE est contraint par la reconstruction et découvre des axes plus informatifs que la projection topologique de UMAP.</p>
<p style="margin: 0;">Le gain de MSE (z=2 → z=3) est attendu mais doit être interprété avec prudence : plus de dimensions = meilleure reconstruction, mais aussi plus de paramètres et moins de contrainte de compression. L'intérêt scientifique réside dans l'<em>interprétabilité</em> du troisième axe, pas uniquement dans le gain de MSE.</p>
</div>

In [ ]:
# ── Courbe d'apprentissage ────────────────────────────────────────────────
viz = DimRedVisualizer(figsize=(12, 4), dpi=150, output_dir=FIGURES_DIR)

fig, axes = viz.plot_training_history(
    history,
    save_path=FIGURES_DIR / "ae_training_history.png",
)
plt.show()

<a id="latent"></a>

<div style="background: linear-gradient(135deg, #5A2A5E 0%, #2C6E8F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(90,42,94,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">3 · Espace latent 2D - visualisation et interprétation physique</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
Les mêmes colorations physiques que dans PCA et UMAP sont utilisées pour comparer directement les géométries d’embedding.
</div>

Un encodeur apprend une application non linéaire :

$$
f_\theta: \mathbb{R}^{183} \rightarrow \mathbb{R}^{2}, \quad z=f_\theta(x)
$$

et le décodeur reconstruit :

$$
g_\phi: \mathbb{R}^{2} \rightarrow \mathbb{R}^{183}, \quad \hat x=g_\phi(z)
$$

L’objectif est de compresser l’information spectrale utile dans deux variables latentes tout en conservant une structure physiquement cohérente.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Encodage ──────────────────────────────────────────────────────────────
Z_ae = ae.encode(X)
print(f"Espace latent : {Z_ae.shape}")
print(f"  axe 1 : [{Z_ae[:,0].min():.2f}, {Z_ae[:,0].max():.2f}]")
print(f"  axe 2 : [{Z_ae[:,1].min():.2f}, {Z_ae[:,1].max():.2f}]")

In [ ]:
# ── Grille de visualisations ──────────────────────────────────────────────
fig, axes = viz.plot_ae_embedding(
    Z_ae, meta, y,
    save_path=FIGURES_DIR / "ae_latent_grid.png",
)
plt.show()

### 3.1 · Espace latent zoomé — population stellaire

La figure précédente a une échelle globale dominée par quelques **points extrêmes** (4 QSOs, certaines galaxies atypiques) que l'autoencodeur entraîné sur des étoiles ne peut pas reconstruire — il les repousse loin de la région stellaire dans l'espace latent. Cela illustre une propriété utile : l'AE agit implicitement comme un **détecteur d'anomalies**.

Pour visualiser la structure interne de la population stellaire, on zoome sur la région principale (percentiles 1–99).

In [ ]:
# ── Espace latent zoomé (population stellaire principale) ───────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Calcul des bornes robustes (percentile 0.5 / 99.5)
x_lo, x_hi = np.percentile(Z_ae[:, 0], [0.5, 99.5])
y_lo, y_hi = np.percentile(Z_ae[:, 1], [0.5, 99.5])
pad_x = (x_hi - x_lo) * 0.10
pad_y = (y_hi - y_lo) * 0.10

CLASS_PALETTE = {
    "STAR": "#4C72B0", "GALAXY": "#DD8452",
    "QSO": "#55A868",  "UNKNOWN": "#8172B3",
}
CLASS_MARKER  = {"STAR": "o", "GALAXY": "s", "QSO": "^", "UNKNOWN": "x"}
PHYS_PARAMS   = [
    ("T_eff (K)",    "teff_gspphot", "plasma"),
    ("[Fe/H]",       "mh_gspphot",   "RdYlBu_r"),
    ("log g",        "logg_gspphot", "viridis"),
]

fig, axes = plt.subplots(1, 1 + len(PHYS_PARAMS), figsize=(22, 5), dpi=150)

# Panneau 1 : coloration par classe
ax = axes[0]
for cls in np.unique(y):
    mask = y == cls
    ax.scatter(
        Z_ae[mask, 0], Z_ae[mask, 1],
        c=CLASS_PALETTE.get(cls, "#888"),
        marker=CLASS_MARKER.get(cls, "o"),
        s=2, alpha=0.45, linewidths=0,
        label=f"{cls} (n={mask.sum()})", rasterized=True,
    )
ax.set_xlim(x_lo - pad_x, x_hi + pad_x)
ax.set_ylim(y_lo - pad_y, y_hi + pad_y)
ax.legend(markerscale=4, fontsize=8, framealpha=0.9)
ax.set_xlabel("Latent axe 1", fontsize=10)
ax.set_ylabel("Latent axe 2", fontsize=10)
ax.set_title("Type spectral", fontsize=11)
ax.grid(True, alpha=0.25)

# Panneaux 2-4 : paramètres physiques
for ax_i, (label, col, cmap) in zip(axes[1:], PHYS_PARAMS):
    if col not in meta.columns:
        ax_i.axis("off")
        continue
    vals  = meta[col].values.astype(float)
    valid = np.isfinite(vals)
    if (~valid).any():
        ax_i.scatter(
            Z_ae[~valid, 0], Z_ae[~valid, 1],
            c="#e0e0e0", s=1, alpha=0.15, rasterized=True, linewidths=0,
        )
    sc = ax_i.scatter(
        Z_ae[valid, 0], Z_ae[valid, 1],
        c=vals[valid], cmap=cmap,
        vmin=np.nanpercentile(vals[valid], 2),
        vmax=np.nanpercentile(vals[valid], 98),
        s=2, alpha=0.55, linewidths=0, rasterized=True,
    )
    plt.colorbar(sc, ax=ax_i, fraction=0.046, pad=0.04, label=label).ax.tick_params(labelsize=8)
    ax_i.set_xlim(x_lo - pad_x, x_hi + pad_x)
    ax_i.set_ylim(y_lo - pad_y, y_hi + pad_y)
    ax_i.set_xlabel("Latent axe 1", fontsize=10)
    ax_i.set_title(label, fontsize=11)
    ax_i.grid(True, alpha=0.25)

fig.suptitle(
    "Autoencodeur — Espace latent zoomé (population stellaire, P0.5–P99.5)\n"
    "LAMOST DR5 × Gaia DR3",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
zoom_path = FIGURES_DIR / "ae_latent_grid_zoomed.png"
fig.savefig(zoom_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"\n✓ Figure zoomée : {zoom_path}")
print(f"  Fenêtre : x=[{x_lo:.2f}, {x_hi:.2f}]  y=[{y_lo:.2f}, {y_hi:.2f}]")
n_outliers = ((Z_ae[:, 0] < x_lo) | (Z_ae[:, 0] > x_hi) |
              (Z_ae[:, 1] < y_lo) | (Z_ae[:, 1] > y_hi)).sum()
print(f"  Outliers exclus de la fenêtre : {n_outliers} / {len(Z_ae)} "
      f"({100 * n_outliers / len(Z_ae):.2f}%)")

### 3.bis.2 · Carte de densité KDE — populations stellaires dans l'espace latent

<div style="
    background: linear-gradient(135deg, rgba(90,42,94,0.10), rgba(44,110,143,0.10));
    border-left: 6px solid #5A2A5E;
    border-radius: 10px;
    padding: 12px 14px;
    margin-bottom: 10px;
">
Le scatter plot coloré par T_eff montre la <em>position</em> de chaque étoile. Cette figure montre la <em>densité</em> : où se concentrent les populations. Les crêtes de densité correspondent aux types spectraux dominants dans LAMOST DR5. Les iso-contours délimitent les régions où 68%, 95% et 99% des étoiles sont concentrées.
</div>

In [ ]:
# ── Carte de densité KDE de l'espace latent AE ───────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import gaussian_kde

# Filtrer les étoiles STAR uniquement (population principale)
star_mask = (y == "STAR")
Z_stars   = Z_ae[star_mask]
teff_stars = meta["teff_gspphot"].values[star_mask].astype(float)
logg_stars = meta["logg_gspphot"].values[star_mask].astype(float)
mh_stars   = meta["mh_gspphot"].values[star_mask].astype(float)

# Sous-échantillonnage pour la KDE (scipy KDE est O(N²))
N_KDE = min(8000, len(Z_stars))
rng_kde = np.random.default_rng(42)
idx_kde = rng_kde.choice(len(Z_stars), N_KDE, replace=False)
Z_kde   = Z_stars[idx_kde]

# Estimation KDE
print(f"Calcul KDE sur {N_KDE} étoiles...")
kde = gaussian_kde(Z_kde.T, bw_method="scott")

# Grille d'évaluation (percentiles pour éviter les extrêmes)
p1, p99 = np.percentile(Z_stars, [1, 99], axis=0)
margin = (p99 - p1) * 0.15
x_grid = np.linspace(p1[0] - margin[0], p99[0] + margin[0], 200)
y_grid = np.linspace(p1[1] - margin[1], p99[1] + margin[1], 200)
XX, YY = np.meshgrid(x_grid, y_grid)
positions = np.vstack([XX.ravel(), YY.ravel()])
ZZ = kde(positions).reshape(XX.shape)

# ── Niveaux de contours robustes (percentiles sur la grille ZZ) ───────────
zz_flat = ZZ.ravel()
zz_flat = zz_flat[zz_flat > 0]
if len(zz_flat) == 0:
    levels = []
else:
    l99 = float(np.percentile(zz_flat, 1))   # contour englobant 99% de la densité
    l95 = float(np.percentile(zz_flat, 5))   # contour englobant 95%
    l68 = float(np.percentile(zz_flat, 32))  # contour englobant 68%
    raw = sorted(set([l99, l95, l68]))        # déduplique + trie (exigé par matplotlib)
    levels = raw if len(raw) >= 2 else []

# ── Figure 3 panneaux : densité + T_eff + log g + [Fe/H] ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=150)

PHYS = [
    (teff_stars, "T_eff (K)",  "plasma",  "Densité + T_eff"),
    (logg_stars, "log g",      "viridis", "Densité + log g"),
    (mh_stars,   "[Fe/H]",     "RdYlBu",  "Densité + [Fe/H]"),
]

for ax, (param, label, cmap, title) in zip(axes, PHYS):
    valid = np.isfinite(param)
    Z_v   = Z_stars[valid]
    P_v   = param[valid]

    # Fond KDE en niveaux de gris
    ax.contourf(XX, YY, ZZ, levels=20, cmap="Greys", alpha=0.45, zorder=1)

    # Iso-contours 68 / 95 / 99%
    if levels:
        # Adapter les couleurs/styles au nombre de niveaux disponibles
        n_lv = len(levels)
        colors_lv    = ["#4A4A4A", "#2A2A2A", "#0A0A0A"][-n_lv:]
        linewidths_lv = [0.8, 1.2, 1.8][-n_lv:]
        linestyles_lv = ["--", "-", "-"][-n_lv:]
        cs = ax.contour(XX, YY, ZZ, levels=levels,
                        colors=colors_lv,
                        linewidths=linewidths_lv,
                        linestyles=linestyles_lv,
                        zorder=2)
        level_names = ["99%", "95%", "68%"][-n_lv:]
        fmt_dict = {lv: nm for lv, nm in zip(levels, level_names)}
        ax.clabel(cs, fmt=fmt_dict, fontsize=7, inline=True)

    # Points colorés par paramètre physique
    N_SHOW = min(5000, len(Z_v))
    idx_s  = rng_kde.choice(len(Z_v), N_SHOW, replace=False)
    sc = ax.scatter(
        Z_v[idx_s, 0], Z_v[idx_s, 1],
        c=P_v[idx_s], cmap=cmap,
        s=1.2, alpha=0.55, rasterized=True,
        vmin=float(np.percentile(P_v, 2)),
        vmax=float(np.percentile(P_v, 98)),
        zorder=3,
    )
    plt.colorbar(sc, ax=ax, label=label, fraction=0.03, pad=0.02)
    ax.set_xlabel("AE axe 1", fontsize=9)
    ax.set_ylabel("AE axe 2", fontsize=9)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlim(p1[0] - margin[0], p99[0] + margin[0])
    ax.set_ylim(p1[1] - margin[1], p99[1] + margin[1])
    ax.grid(True, alpha=0.15)

fig.suptitle(
    "Carte de densité KDE de l'espace latent AE z=2 — population stellaire LAMOST DR5\n"
    "Iso-contours : 68% / 95% / 99% de la densité",
    fontsize=12, fontweight="bold", y=1.02,
)
plt.tight_layout()
kde_path = FIGURES_DIR / "ae_latent_kde.png"
fig.savefig(kde_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"✓ Sauvegardée : {kde_path}")

<div style="
    background: linear-gradient(135deg, rgba(90,42,94,0.10), rgba(44,110,143,0.10));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 12px 14px;
    margin: 6px 0 12px 0;
">
<h4 style="margin: 0 0 8px 0; color: #2C6E8F;">Lecture scientifique — Topographie de l'espace latent</h4>
<p style="margin: 0 0 6px 0;">Les crêtes de densité révèlent la structure intrinsèque du dataset LAMOST DR5 : une <strong>crête principale</strong> correspondant à la séquence principale (naines F/G/K), des zones de <strong>faible densité</strong> aux extrêmes (étoiles M froides, étoiles A chaudes — moins représentées dans le catalogue), et des <strong>excursions</strong> au-delà du contour 99% signalant les objets atypiques.</p>
<p style="margin: 0;">La superposition T_eff / log g / [Fe/H] sur la même géométrie permet de vérifier que les axes latents encodent bien des informations physiques complémentaires et non redondantes.</p>
</div>

### Interprétation — Espace latent et détection d'anomalies

La version zoomée révèle la structure interne de la population stellaire :
- **Axe latent 2** encode principalement la **température** (ρ Spearman = -0.76 avec T_eff) — le gradient chaud/froid est clairement visible.
- **Axe latent 1** encode une combinaison de métallicité et de stade évolutif (ρ ≈ 0.27 avec [Fe/H]), mais de façon moins nette que PC1 de la PCA.

Les axes latents sont **arbitraires en orientation** (signe et ordre), contrairement aux composantes PCA ordonnées par variance décroissante. C'est une limitation fondamentale des autoencodeurs non supervisés.

**Propriété émergente — détection d'anomalies** : les quelques galaxies et QSOs sont repoussés à des coordonnées latentes extrêmes (jusqu'à y≈88 pour les QSOs). L'AE, entraîné quasi-exclusivement sur des étoiles, ne peut pas reconstruire les spectres non-stellaires → il les projette hors de la distribution apprise. Un seuil simple sur la distance à l'origine dans l'espace latent suffirait à les détecter sans supervision.

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.10), rgba(90,42,94,0.10));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 14px 16px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 8px 0; color: #2C6E8F;">Corrélations espace latent et paramètres physiques (Spearman)</h4>
Comparer ces corrélations avec PC1/PC2 permet d’évaluer ce que l’autoencodeur préserve (ou transforme) de la structure physique stellaire.
</div>

Le coefficient de Spearman mesure la monotonicité entre deux variables de rang :

$$
\rho_s = 1 - \frac{6\sum_i d_i^2}{n(n^2-1)}
$$

où $d_i$ est la différence entre les rangs de deux observations.

### Interprétation
- Une forte $|\rho_s|$ entre un axe latent et T_eff ou BP-RP indique une capture efficace d’un gradient astrophysique majeur.
- Une redistribution de l’information entre axe 1 et axe 2 est normale en non linéaire : les axes latents ne sont pas directement comparables aux composantes PCA.

In [ ]:
# Corrélations espace latent vs paramètres physiques
from scipy.stats import spearmanr

phys_cols = ["teff_gspphot", "logg_gspphot", "mh_gspphot", "bp_rp"]
phys_labels = ["T_eff", "log g", "[Fe/H]", "G_BP-G_RP"]

print("Corrélations Spearman : Espace latent AE ↔ Paramètres physiques")
print("-" * 60)
print(f"{'Paramètre':<15} {'Axe latent 1':>14} {'Axe latent 2':>14}")
print("-" * 60)

for col, lbl in zip(phys_cols, phys_labels):
    if col not in meta.columns:
        continue
    vals = meta[col].values.astype(float)
    valid = np.isfinite(vals)
    r1, _ = spearmanr(Z_ae[valid, 0], vals[valid])
    r2, _ = spearmanr(Z_ae[valid, 1], vals[valid])
    print(f"{lbl:<15} {r1:>+14.3f} {r2:>+14.3f}")

print("-" * 60)

# Comparaison directe avec PC1/PC2
print("\nRappel — PC1/PC2 (notebook 01) :")
print(f"{'Paramètre':<15} {'PC1':>14} {'PC2':>14}")
print("-" * 60)
corr_pca = pca.correlations_with_params(meta[phys_cols], scores=scores_pca, n_pcs=2)
for col, lbl in zip(phys_cols, phys_labels):
    if col not in corr_pca.columns:
        continue
    print(f"{lbl:<15} {corr_pca.loc['PC1', col]:>+14.3f} {corr_pca.loc['PC2', col]:>+14.3f}")

<a id="recon"></a>

<div style="background: linear-gradient(135deg, #2C6E8F 0%, #3D405B 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(44,110,143,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">4 · Qualité de reconstruction - autoencodeur vs PCA</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
<strong>4.1 Exemples de reconstruction</strong>
</div>

Pour chaque objet, l’erreur quadratique moyenne est :

$$
\mathrm{MSE}_i = \frac{1}{D}\sum_{j=1}^{D}(x_{ij}-\hat x_{ij})^2
$$

La comparaison AE vs PCA permet de mesurer le gain de non-linéarité à dimension fixée (2D), puis de situer ce gain face à une PCA plus riche (n composantes pour 95% de variance).

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #2C6E8F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Reconstruction de l'ensemble ─────────────────────────────────────────
X_recon = ae.reconstruct(X)

print(f"MSE globale autoencodeur (latent_dim=2) : {ae.reconstruction_mse(X):.5f}")
print(f"MSE globale PCA (n=2)                   : "
      f"{pca.reconstruction_error(X, n_components=2).mean():.5f}")
print(f"MSE globale PCA (n={n_pcs_95})                  : "
      f"{pca.reconstruction_error(X, n_components=n_pcs_95).mean():.5f}")

In [ ]:
# ── Exemples visuels : original vs reconstruit ────────────────────────────
fig, axes = viz.plot_ae_reconstruction(
    X, X_recon, feature_names,
    n_samples=5,
    y=y,
    save_path=FIGURES_DIR / "ae_reconstruction_examples.png",
)
plt.show()

### 4.ter · Reconstruction originale vs reconstruite — profil par type spectral

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin-bottom: 10px;
">
Pour chaque type spectral (A, F, G, K, M), on calcule le profil moyen des <strong>features clés</strong> pour les spectres originaux et leurs versions reconstruites par l'AE. L'écart entre les deux courbes révèle où le modèle est fidèle et où il approxime — une validation visuelle de la qualité de reconstruction.
</div>

In [ ]:
# ── Reconstruction par type spectral — types depuis NB02 (XGBoost) ───────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

_RECON_FEATURES = [
    "feature_Hα_eq_width", "feature_Hβ_eq_width", "feature_Hδ_eq_width",
    "feature_CaIIK_eq_width", "feature_CaII_H_eq_width",
    "feature_Mg_b_eq_width", "feature_Na_D_eq_width", "feature_FeH_proxy",
    "feature_Teff_proxy", "feature_synthetic_BV",
    "feature_flux_ratio_blue_red", "feature_molecular_TiO_7050",
]
_RECON_LABELS = [
    "Hα EW", "Hβ EW", "Hδ EW",
    "Ca II K", "Ca II H", "Mg b", "Na D", "FeH proxy",
    "T_eff proxy", "BV synthét.", "Flux B/R", "TiO 7050",
]

_valid_pairs = [
    (label, feat)
    for label, feat in zip(_RECON_LABELS, _RECON_FEATURES)
    if feat in feature_names
]
_labels_ok = [p[0] for p in _valid_pairs]
_feat_idx  = [feature_names.index(p[1]) for p in _valid_pairs]

X_recon = ae.reconstruct(X)

# ── Récupération des types spectraux XGBoost depuis le joblib NB02 ────────
# NB02 a déjà prédit A/F/G/K/M et sauvegardé dans phy3500_embeddings_latest
import joblib

_emb_path = Path(MODELS_DIR).parent / "runs" / "phy3500_umap_tsne" / "phy3500_embeddings_latest.joblib"
_xgb_ok = False

if _emb_path.exists():
    _emb = joblib.load(str(_emb_path))
    if "y_pred_xgb" in _emb:
        spectral_types = np.array(_emb["y_pred_xgb"])
        _xgb_ok = True
        print(f"✓ Types XGBoost chargés depuis NB02 : "
              f"{dict(zip(*np.unique(spectral_types, return_counts=True)))}")
    else:
        print(f"⚠  Clé 'y_pred_xgb' absente dans le joblib.")
        print(f"   Clés disponibles : {list(_emb.keys())}")
else:
    print(f"⚠  Joblib NB02 introuvable : {_emb_path}")

if not _xgb_ok:
    # Fallback : sous-type LAMOST depuis meta (si disponible)
    if "subclass" in meta.columns:
        spectral_types = meta["subclass"].str[0].str.upper().values
        spectral_types = np.where(np.isin(spectral_types, ["A","F","G","K","M"]),
                                  spectral_types, "?")
        print("⚠  Fallback sur subclass LAMOST (premier caractère)")
    else:
        spectral_types = y
        print("⚠  Fallback sur classe LAMOST (STAR/GALAXY/QSO)")

# ── Figure : grille types × features clés ────────────────────────────────
TYPE_ORDER  = ["A", "F", "G", "K", "M"]
TYPE_COLORS = {
    "A": "#5DCAA5", "F": "#F5A623", "G": "#E8593C",
    "K": "#B07DB8", "M": "#3B8BD4",
}

# Filtrer sur les types qui existent dans les prédictions
TYPE_ORDER = [t for t in TYPE_ORDER if np.sum(spectral_types == t) >= 10]

if not TYPE_ORDER:
    print("⚠  Aucun type spectral A/F/G/K/M — vérifier que NB02 a bien tourné.")
else:
    n_types = len(TYPE_ORDER)
    x_pos   = np.arange(len(_feat_idx))
    w = 0.35

    fig, axes = plt.subplots(1, n_types, figsize=(4 * n_types, 6),
                              dpi=150, sharey=True)
    if n_types == 1:
        axes = [axes]

    for ax, stype in zip(axes, TYPE_ORDER):
        mask = (spectral_types == stype)
        orig_mean  = X[mask][:, _feat_idx].mean(axis=0)
        recon_mean = X_recon[mask][:, _feat_idx].mean(axis=0)
        col = TYPE_COLORS.get(stype, "#888")

        ax.bar(x_pos - w/2, orig_mean,  w, color=col, alpha=0.85,
               label="Original",    edgecolor="white", linewidth=0.4)
        ax.bar(x_pos + w/2, recon_mean, w, color=col, alpha=0.40,
               label="Reconstruit", edgecolor=col, linewidth=1.2)

        for xi, (o, r) in enumerate(zip(orig_mean, recon_mean)):
            ax.plot([xi, xi], [o, r], color="#333", lw=0.8, ls=":", zorder=4)

        ax.axhline(0, color="gray", lw=0.6, alpha=0.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(_labels_ok, rotation=45, ha="right", fontsize=7)
        ax.set_title(f"Type {stype}\nn={mask.sum():,}",
                     fontsize=11, fontweight="bold", color=col)
        ax.grid(axis="y", alpha=0.2)
        if ax == axes[0]:
            ax.set_ylabel("Score standardisé moyen (σ)", fontsize=9)
            ax.legend(fontsize=8, loc="upper right")

    src_label = "XGBoost NB02" if _xgb_ok else "Subclass LAMOST"
    fig.suptitle(
        f"Profil moyen original vs reconstruit par type spectral ({src_label})\n"
        "(scores standardisés — traits pointillés = résidu AE)",
        fontsize=12, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    recon_type_path = FIGURES_DIR / "ae_recon_by_spectral_type.png"
    fig.savefig(recon_type_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"✓ Sauvegardée : {recon_type_path}")

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 6px 0 12px 0;
">
<h4 style="margin: 0 0 8px 0;">Lecture par type spectral</h4>
<p style="margin: 0 0 6px 0;"><strong>Types A/F (chauds) :</strong> Hα et Hβ très positifs, Ca II faibles ou négatifs — l'AE devrait bien reconstruire car ces signatures sont fortes et cohérentes.</p>
<p style="margin: 0 0 6px 0;"><strong>Types G/K (solaires) :</strong> profil équilibré, signatures modérées sur tous les indices. Zone la plus représentée dans LAMOST → reconstruction la plus précise attendue.</p>
<p style="margin: 0;"><strong>Type M (froid) :</strong> TiO très fort, Balmer très faibles, Ca II modérés. L'AE est moins entraîné sur ces étoiles (sous-représentées) → résidus potentiellement plus grands sur TiO.</p>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 8px 0;
">
<strong>4.2 Comparaison MSE autoencodeur vs PCA</strong>
</div>

In [ ]:
# ── Comparaison AE vs PCA ─────────────────────────────────────────────────
comparison_df = ae.compare_with_pca(
    X, pca,
    n_pcs_list=[1, 2, 3, 5, 8, 10, 15, 20, 30, 50],
)
print(comparison_df.to_string(index=False))

fig, ax = viz.plot_ae_vs_pca(
    comparison_df,
    save_path=FIGURES_DIR / "ae_vs_pca_mse.png",
)
plt.show()

### Interprétation — AE@2 ≈ PCA@10 : la valeur de la non-linéarité

C'est le résultat central de ce notebook :

| Méthode | Dimensions | MSE reconstruction |
|---------|-----------|-------------------|
| PCA     | 2         | 0.7317            |
| PCA     | 5         | 0.6290            |
| PCA     | 8         | 0.5722            |
| **AE**  | **2**     | **0.5411**        |
| PCA     | 10        | 0.5430            |
| PCA     | 15        | 0.4803            |

L'autoencodeur à **2 dimensions latentes** reconstruit les spectres aussi bien qu'une **PCA à 10 composantes**. Autrement dit, la non-linéarité permet d'encoder 5× plus d'information dans le même nombre de dimensions.

Cela s'explique physiquement : la séquence stellaire n'est pas un sous-espace euclidien — c'est une **variété courbée** dans l'espace des features. La PCA doit utiliser de nombreux axes linéaires pour approximer cette courbure, là où l'AE la capture directement.

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 8px 0;
">
<strong>4.3 Distribution des erreurs par classe</strong>
</div>

In [ ]:
# ── Résumé par classe ─────────────────────────────────────────────────────
summary = ae.reconstruction_summary(X, y=y)
print("\nErreurs de reconstruction par classe :")
print(summary.to_string(index=False))

# Distribution
mse_per = ae.reconstruction_mse(X, per_sample=True)

fig, ax = viz.plot_ae_error_distribution(
    mse_per, y,
    save_path=FIGURES_DIR / "ae_error_distribution.png",
)
plt.show()

<a id="recon-family"></a>

<div style="background: linear-gradient(135deg, #2C6E8F 0%, #3D405B 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(44,110,143,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">4.bis · Erreur de reconstruction par famille spectroscopique</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.10), rgba(61,64,91,0.10));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 12px 14px;
    margin-bottom: 10px;
">
<p style="margin: 0 0 8px 0;">La symétrie parfaite avec §4.bis de NB01 : au lieu de demander <em>quelle famille drive PC1</em>, on demande <em>quelle famille l'AE reconstruit-il le mieux</em>. Les familles avec les MSE les plus basses sont celles que le modèle a internalisées avec le plus de précision — ce sont les features les plus structurantes pour la reconstruction.</p>
<p style="margin: 0;"><strong>Hypothèse :</strong> Balmer et Ca II (forts et systématiques) → MSE basse. TiO et s-process (rares, seulement dans K/M et exotiques) → MSE élevée.</p>
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #2C6E8F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Erreur de reconstruction par famille spectroscopique ─────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re

# ── Réutilise la même définition de familles que NB01 ────────────────────
_FAM_PATTERNS = {
    "Balmer\n(Hα/β/γ/δ/ε/8/9/10)": [
        r"H[αβγδε]|Halpha|Hbeta|Hgamma|Hdelta|Hepsilon|H8|H9|H10"
        r"|feature_H[89]|feature_H10|balmer|paschen"
    ],
    "Calcium\n(Ca II H&K + triplet)": [r"CaII|CaH|CaK|Ca_8|Ca_trip|feature_Ca"],
    "Magnésium\n(Mg b + triplet)":    [r"Mg_b|Mg_5|MgH|Mg_trip|feature_Mg"],
    "Fer & métaux\n(Fe, Cr, Ni, Co, V, Al)": [
        r"feature_Fe|feature_Cr|feature_Ni|feature_Co|feature_V_"
        r"|feature_Al|iron_peak|metal_index|metal_poor|FeH_proxy|alpha_Fe|alpha_el"
    ],
    "Sodium\n(Na D)":                 [r"Na_D|feature_Na"],
    "TiO & moléc.\n(TiO, VO, CH, CN)": [r"TiO|VO_|molecular|feature_Ti|CNO|CN_"],
    "s-process\n(Sr, Ba)": [r"feature_Sr|feature_Ba|s_process|ratio_Ba|ratio_Sr"],
    "Continuum\n(pente, couleur)": [
        r"continuum|slope|break_4000|flux_ratio|synthetic_BV|UV_excess"
        r"|curvature|color_|feature_cont"
    ],
    "Profils\n(ailes, asymétrie)": [
        r"asymmetr|wing|kurtosis|skewness|core_width|base_width|depth"
        r"|profile_shape|avg_line|rotation"
    ],
    "Indices Lick\n& synthétiques": [
        r"feature_index|Dn4000|G4300|Hbeta_index|NaD_Lick|TiO_1_Lick"
        r"|ratio_EW|EW_CaHK|contrast_metals"
    ],
    "Détection\n(match_*, present)": [r"match_|feature_.*_present"],
    "Autres":                         [r".*"],
}

_FAM_COLORS = [
    "#E8593C", "#3B8BD4", "#4C9B6F", "#B07DB8", "#F5A623",
    "#2E86AB", "#D4A853", "#7F8FA6", "#C06C84", "#6CAE75",
    "#A3B4C5", "#CCCCCC",
]
_FAM_COLOR_MAP = {f: _FAM_COLORS[i % len(_FAM_COLORS)]
                  for i, f in enumerate(_FAM_PATTERNS.keys())}

def _assign_family(feat):
    for fam, pats in _FAM_PATTERNS.items():
        for p in pats:
            if re.search(p, feat, re.IGNORECASE):
                return fam
    return "Autres"

# ── Calcul MSE feature-par-feature ───────────────────────────────────────
X_recon_ae = ae.reconstruct(X)           # (N, 183) standardisé
mse_per_feat = np.mean((X - X_recon_ae) ** 2, axis=0)  # (183,)

df_feat_mse = pd.DataFrame({
    "feature": feature_names,
    "mse":     mse_per_feat,
    "family":  [_assign_family(f) for f in feature_names],
})

# Agrégation par famille
df_fam = (
    df_feat_mse.groupby("family")["mse"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "mse_mean", "std": "mse_std", "count": "n_feats"})
    .reset_index()
    .sort_values("mse_mean")
)

print("Erreur de reconstruction par famille (MSE moyen — features standardisées)")
print("=" * 70)
for _, row in df_fam.iterrows():
    bar = "█" * int(row['mse_mean'] * 30)
    fam_short = row['family'].replace("\n", " ")
    print(f"  {fam_short:38s}  {row['mse_mean']:.4f} ± {row['mse_std']:.4f}  ({row['n_feats']:3.0f} feat)  {bar}")

# ── Figure : barres horizontales + scatter (MSE vs contribution PC1) ─────
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=150)

# Panneau gauche : MSE par famille (barh)
ax = axes[0]
fams   = df_fam["family"].tolist()
mses   = df_fam["mse_mean"].tolist()
stds   = df_fam["mse_std"].tolist()
colors = [_FAM_COLOR_MAP.get(f, "#CCCCCC") for f in fams]

bars = ax.barh(
    [f.replace("\n", " ") for f in fams],
    mses, xerr=stds,
    color=colors, edgecolor="white", linewidth=0.5,
    error_kw={"elinewidth": 1, "capsize": 3, "ecolor": "#666"},
)
ax.set_xlabel("MSE de reconstruction (features standardisées)", fontsize=10)
ax.set_title(
    "MSE de reconstruction par famille\n"
    "(barres triées par MSE croissante — mieux reconstruit à gauche)",
    fontsize=11, fontweight="bold",
)
ax.axvline(float(np.mean(mse_per_feat)), color="#333", lw=1.2, ls="--",
           label=f"MSE global = {np.mean(mse_per_feat):.4f}")
ax.legend(fontsize=9)
ax.grid(axis="x", alpha=0.25)

# Panneau droit : scatter MSE vs n_features dans la famille
ax2 = axes[1]
nf = df_fam["n_feats"].tolist()
for fam, mse, n, c in zip(fams, mses, nf, colors):
    ax2.scatter(n, mse, s=120, color=c, edgecolors="white", linewidths=0.8,
                zorder=3)
    ax2.annotate(
        fam.replace("\n", " ")[:22],
        (n, mse),
        textcoords="offset points", xytext=(6, 0),
        fontsize=6.5, color="#333",
    )
ax2.set_xlabel("Nombre de features dans la famille", fontsize=10)
ax2.set_ylabel("MSE de reconstruction moyenne", fontsize=10)
ax2.set_title(
    "MSE vs taille de la famille\n"
    "(familles larges = plus de redondance = mieux reconstruites ?)",
    fontsize=11, fontweight="bold",
)
ax2.grid(True, alpha=0.25)

plt.tight_layout()
fam_mse_path = FIGURES_DIR / "ae_recon_error_by_family.png"
fig.savefig(fam_mse_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"\n✓ Sauvegardée : {fam_mse_path}")

# ── Top 10 features les mieux / moins bien reconstruites ─────────────────
print("\nTop 10 features les mieux reconstruites (MSE la plus basse) :")
print(df_feat_mse.nsmallest(10, "mse")[["feature", "family", "mse"]].to_string(index=False))
print("\nTop 10 features les moins bien reconstruites (MSE la plus haute) :")
print(df_feat_mse.nlargest(10, "mse")[["feature", "family", "mse"]].to_string(index=False))

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.10), rgba(61,64,91,0.10));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 12px 14px;
    margin: 6px 0 12px 0;
">
<h4 style="margin: 0 0 8px 0; color: #2C6E8F;">Interprétation — Ce que l'AE apprend vraiment</h4>
<p style="margin: 0 0 6px 0;">Les familles avec une <strong>MSE basse</strong> sont celles que l'autoencodeur reconstruit fidèlement : il a appris à les prédire précisément depuis l'espace latent. Ce sont les features les plus <em>redondantes et structurées</em> dans le dataset.</p>
<p style="margin: 0 0 6px 0;">Les familles avec une <strong>MSE haute</strong> sont soit rares (s-process, présent seulement dans quelques % des étoiles), soit très variables d'une étoile à l'autre sans pattern simple. L'AE ne peut pas les reconstruire précisément depuis 2 dimensions latentes.</p>
<p style="margin: 0;">Cette analyse complète la décomposition des loadings PCA (NB01) : PCA identifie les features qui <em>varient ensemble</em>, l'AE identifie les features qu'il <em>peut modéliser</em> — les deux angles se recoupent et se valident mutuellement.</p>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 8px 0;
">
<strong>4.4 · Distribution des erreurs — version corrigée (axe logarithmique)</strong>
</div>

La figure précédente avait une échelle linéaire dominée par les QSOs (MSE médiane ≈ 30), rendant la distribution des étoiles invisible. On utilise ici un axe x logarithmique pour visualiser toutes les classes simultanément.

In [ ]:
# ── Distribution des erreurs — version log scale ──────────────────────────
import matplotlib.pyplot as plt
import numpy as np

mse_per = ae.reconstruction_mse(X, per_sample=True)

CLASS_COLORS = {
    "STAR": "#4C72B0", "GALAXY": "#DD8452",
    "QSO": "#55A868",  "UNKNOWN": "#8172B3",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

for ax, xscale in zip(axes, ["linear", "log"]):
    for cls in np.unique(y):
        mask = y == cls
        vals = mse_per[mask]
        med  = np.median(vals)
        color = CLASS_COLORS.get(cls, "gray")
        # bins adaptés à l'échelle
        if xscale == "log":
            bins = np.logspace(
                np.log10(max(vals.min(), 1e-4)), np.log10(vals.max() + 1), 60
            )
        else:
            # Limiter à P99 des étoiles pour l'échelle linéaire
            star_p99 = np.percentile(mse_per[y == "STAR"], 99)
            bins = np.linspace(0, star_p99 * 1.5, 60)
        ax.hist(
            vals, bins=bins, alpha=0.55, color=color, density=True,
            label=f"{cls}  médiane={med:.3f}",
        )
        ax.axvline(med, color=color, lw=1.8, ls="--", alpha=0.9)
    ax.set_xlabel("MSE de reconstruction", fontsize=11)
    ax.set_ylabel("Densité", fontsize=11)
    ax.set_xscale(xscale)
    title = "Échelle linéaire (P99 étoiles)" if xscale == "linear" else "Échelle logarithmique (toutes classes)"
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    "Distribution des erreurs de reconstruction — AE vs classes spectrales",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
err_path = FIGURES_DIR / "ae_error_distribution_logscale.png"
fig.savefig(err_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"\n✓ Sauvegardée : {err_path}")

### Interprétation — L'autoencodeur comme détecteur d'anomalies

La distribution des erreurs révèle une **séparation spectaculaire** entre les classes :

| Classe | n | MSE médiane | Rapport vs étoiles |
|--------|---|------------|--------------------|
| STAR   | 42 862 | 0.281 | ×1 |
| GALAXY | 54 | 1.824 | ×**6.5** |
| QSO    | 4  | 30.06 | ×**107** |

Un simple **seuil sur la MSE de reconstruction** (par exemple MSE > 2.0) permettrait d'isoler les galaxies et QSOs du catalogue stellaire sans aucune supervision ni label. Cette propriété émerge naturellement du fait que l'AE a appris une représentation compressée optimisée pour les étoiles — les autres types d'objets y sont des **outliers** par construction.

> **Application potentielle** : sur le catalogue complet LAMOST DR5 (~9 millions de spectres), ce détecteur pourrait identifier automatiquement les contaminations non-stellaires sans annotation manuelle.

<a id="interp"></a>

<div style="background: linear-gradient(135deg, #3D405B 0%, #5A2A5E 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(61,64,91,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">5 · Continuité de l'espace latent - interpolation</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(61,64,91,0.10), rgba(90,42,94,0.10));
    border-left: 6px solid #3D405B;
    border-radius: 10px;
    padding: 14px 16px;
    margin: 0 0 10px 0;
">
Une interpolation linéaire entre deux points latents doit produire des reconstructions physiquement cohérentes si la géométrie latente est bien apprise.
</div>

Interpolation entre deux codes latents $z_A$ et $z_B$ :

$$
z(\alpha)=(1-\alpha)z_A + \alpha z_B, \quad \alpha\in[0,1]
$$

Si l’espace latent est régulier, les spectres reconstruits le long de la trajectoire doivent évoluer progressivement (sans sauts non physiques majeurs).

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #3D405B; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>


Une propriété clé d'un bon espace latent est sa **continuité** :
interpoler linéairement entre deux points dans l'espace latent doit
produire des reconstructions physiquement cohérentes.

Ici on interpole entre une **étoile froide** (T_eff ~ 4000 K)
et une **étoile chaude** (T_eff ~ 8000 K) et on observe comment
les features reconstruites évoluent.

In [ ]:
# Sélection d'une étoile froide et d'une étoile chaude
teff = meta["teff_gspphot"].values
valid_teff = np.isfinite(teff)

# Étoile froide : T_eff la plus basse dans le top-SNR
cold_candidates = np.where(valid_teff & (teff < 4200))[0]
hot_candidates  = np.where(valid_teff & (teff > 7500))[0]

if len(cold_candidates) == 0 or len(hot_candidates) == 0:
    print("⚠ Pas assez de candidats — utilisation des percentiles 5 et 95")
    cold_candidates = np.where(valid_teff & (teff < np.nanpercentile(teff, 5)))[0]
    hot_candidates  = np.where(valid_teff & (teff > np.nanpercentile(teff, 95)))[0]

# Prend le meilleur SNR dans chaque groupe
snr_r = meta["snr_r"].values if "snr_r" in meta.columns else np.ones(len(X))

idx_cold = cold_candidates[np.argmax(snr_r[cold_candidates])]
idx_hot  = hot_candidates[np.argmax(snr_r[hot_candidates])]

teff_cold = teff[idx_cold]
teff_hot  = teff[idx_hot]
label_a   = f"Froide (T_eff={teff_cold:.0f} K)"
label_b   = f"Chaude (T_eff={teff_hot:.0f} K)"

print(f"Étoile A (froide) : indice {idx_cold}, T_eff={teff_cold:.0f} K")
print(f"Étoile B (chaude) : indice {idx_hot},  T_eff={teff_hot:.0f} K")

<a id="arithmetic"></a>

<div style="background: linear-gradient(135deg, #3D405B 0%, #2C6E8F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(61,64,91,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">5.ter · Arithmetic latente — "word2vec stellaire"</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(61,64,91,0.10), rgba(44,110,143,0.10));
    border-left: 6px solid #3D405B;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<p style="margin: 0 0 8px 0;">En NLP, l'espace latent de word2vec permet l'<strong>arithmétique sémantique</strong> : <em>roi − homme + femme ≈ reine</em>. Si l'espace latent de l'AE encode des propriétés physiques de façon structurée, une opération analogue devrait fonctionner sur les étoiles :</p>
<p style="text-align: center; font-size: 15px; margin: 10px 0;"><strong>z(K géante) − z(K naine) + z(G naine) ≈ z(G géante) ?</strong></p>
<p style="margin: 0;">Si oui, cela démontre que l'espace latent encode séparément la <em>température</em> (K↔G) et la <em>gravité de surface</em> (naine↔géante) sous forme de vecteurs additifs — une preuve de disentanglement partiel.</p>
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #3D405B; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Arithmetic latente — word2vec stellaire ───────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial.distance import cdist

# ── Définition des populations de référence ───────────────────────────────
# On sélectionne des étoiles typiques dans chaque case du diagramme HR
teff_v = meta["teff_gspphot"].values.astype(float)
logg_v = meta["logg_gspphot"].values.astype(float)
mh_v   = meta["mh_gspphot"].values.astype(float)
valid_all = np.isfinite(teff_v) & np.isfinite(logg_v) & (y == "STAR")

def select_population(teff_range, logg_range, n=200, label=""):
    """Sélectionne n étoiles dans une zone T_eff × log g, retourne z_mean."""
    t_lo, t_hi = teff_range
    g_lo, g_hi = logg_range
    mask = (
        valid_all
        & (teff_v >= t_lo) & (teff_v < t_hi)
        & (logg_v >= g_lo) & (logg_v < g_hi)
    )
    n_found = mask.sum()
    if n_found < 20:
        print(f"  ⚠  {label} : seulement {n_found} étoiles trouvées — résultat peu fiable")
    n_sel = min(n, n_found)
    if n_sel == 0:
        return None, 0
    rng_a = np.random.default_rng(42)
    idx_sel = rng_a.choice(np.where(mask)[0], n_sel, replace=False)
    z_mean = Z_ae[idx_sel].mean(axis=0)
    print(f"  {label:20s} : n={n_found:5d} sélectionnés={n_sel:3d} | "
          f"z̄=({z_mean[0]:+.3f}, {z_mean[1]:+.3f})")
    return z_mean, n_found

print("Sélection des populations de référence :")
print("-" * 65)

# Naines (log g > 4.0)
z_K_naine, nK_n = select_population((3800, 5200), (4.0, 5.5), label="K naine")
z_G_naine, nG_n = select_population((5200, 6200), (4.0, 5.5), label="G naine")
z_F_naine, nF_n = select_population((6200, 7500), (4.0, 5.5), label="F naine")

# Géantes (log g < 3.0)
z_K_geante, nK_g = select_population((3800, 5200), (0.5, 3.0), label="K géante")
z_G_geante, nG_g = select_population((5200, 6200), (0.5, 3.0), label="G géante")
z_F_geante, nF_g = select_population((6200, 7500), (0.5, 3.0), label="F géante")

print()

# ── Tests d'arithmétique ──────────────────────────────────────────────────
TESTS = []
if all(z is not None for z in [z_K_naine, z_G_naine, z_K_geante, z_G_geante]):
    TESTS.append({
        "equation":  "K_géante − K_naine + G_naine ≈ G_géante ?",
        "predicted": z_K_geante - z_K_naine + z_G_naine,
        "target":    z_G_geante,
        "label_pred": "Prédit (K_géante−K_naine+G_naine)",
        "label_tgt":  "Réel G_géante",
        "color_pred": "#E8593C",
        "color_tgt":  "#3B8BD4",
    })
if all(z is not None for z in [z_G_naine, z_K_naine, z_G_geante, z_K_geante]):
    TESTS.append({
        "equation":  "G_géante − G_naine + K_naine ≈ K_géante ?",
        "predicted": z_G_geante - z_G_naine + z_K_naine,
        "target":    z_K_geante,
        "label_pred": "Prédit (G_géante−G_naine+K_naine)",
        "label_tgt":  "Réel K_géante",
        "color_pred": "#4C9B6F",
        "color_tgt":  "#B07DB8",
    })
if all(z is not None for z in [z_K_naine, z_F_naine, z_K_geante, z_F_geante]):
    TESTS.append({
        "equation":  "K_géante − K_naine + F_naine ≈ F_géante ?",
        "predicted": z_K_geante - z_K_naine + z_F_naine,
        "target":    z_F_geante,
        "label_pred": "Prédit (K_géante−K_naine+F_naine)",
        "label_tgt":  "Réel F_géante",
        "color_pred": "#F5A623",
        "color_tgt":  "#2E86AB",
    })

print("=" * 65)
print("  RÉSULTATS DE L'ARITHMÉTIQUE LATENTE")
print("=" * 65)
for t in TESTS:
    pred = t["predicted"]
    tgt  = t["target"]
    dist = float(np.linalg.norm(pred - tgt))

    # Distance de référence (K_naine ↔ G_géante — cas extrême)
    if z_K_naine is not None and z_G_geante is not None:
        dist_ref = float(np.linalg.norm(z_K_naine - z_G_geante))
        ratio = dist / dist_ref * 100
    else:
        ratio = float("nan")

    print(f"\n  {t['equation']}")
    print(f"    Prédit  : ({pred[0]:+.4f}, {pred[1]:+.4f})")
    print(f"    Réel    : ({tgt[0]:+.4f},  {tgt[1]:+.4f})")
    print(f"    Distance Euclid. prédic/réel : {dist:.4f}")
    print(f"    (= {ratio:.1f}% de la distance K_naine↔G_géante)")
    verdict = "✓ ANALOGIE FONCTIONNELLE" if ratio < 30 else (
        "~ Partielle" if ratio < 60 else "✗ Pas d'analogie"
    )
    print(f"    Verdict : {verdict}")

# ── Figure : espace latent + vecteurs arithmétiques ──────────────────────
fig, ax = plt.subplots(figsize=(10, 8), dpi=150)

# Background
sc = ax.scatter(
    Z_ae[valid_all, 0], Z_ae[valid_all, 1],
    c=teff_v[valid_all], cmap="plasma",
    s=0.5, alpha=0.2, rasterized=True, zorder=1,
)
plt.colorbar(sc, ax=ax, label="T_eff Gaia (K)", fraction=0.025)

# Populations de référence
POPS = [
    (z_K_naine,  "K naine",  "#B07DB8", "o"),
    (z_G_naine,  "G naine",  "#E8593C", "o"),
    (z_F_naine,  "F naine",  "#F5A623", "o"),
    (z_K_geante, "K géante", "#B07DB8", "^"),
    (z_G_geante, "G géante", "#E8593C", "^"),
    (z_F_geante, "F géante", "#F5A623", "^"),
]
for z, label, col, marker in POPS:
    if z is None:
        continue
    ax.scatter(z[0], z[1], s=200, c=col, marker=marker,
               edgecolors="white", linewidths=1.5, zorder=5)
    ax.annotate(label, (z[0], z[1]),
                textcoords="offset points", xytext=(8, 4),
                fontsize=9, fontweight="bold", color=col,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8))

# Vecteurs arithmétiques
arrow_kw = dict(arrowstyle="->", color="white", lw=1.5)
COLORS_VEC = ["#FFD700", "#00FF88", "#FF6B9D"]
for ti, t in enumerate(TESTS):
    col_v = COLORS_VEC[ti % len(COLORS_VEC)]
    pred = t["predicted"]
    tgt  = t["target"]
    # Point prédit
    ax.scatter(pred[0], pred[1], s=180, c=col_v, marker="*",
               edgecolors="black", linewidths=0.8, zorder=6)
    # Flèche prédit → réel
    ax.annotate(
        "", xy=(tgt[0], tgt[1]), xytext=(pred[0], pred[1]),
        arrowprops=dict(arrowstyle="->", color=col_v, lw=1.8, ls="--"),
        zorder=7,
    )
    dist = np.linalg.norm(pred - tgt)
    ax.annotate(
        f"d={dist:.3f}",
        xy=((pred[0] + tgt[0]) / 2, (pred[1] + tgt[1]) / 2),
        fontsize=7.5, color=col_v,
        bbox=dict(boxstyle="round,pad=0.15", fc="black", alpha=0.7),
        zorder=8,
    )

# Légende
legend_handles = [
    mpatches.Patch(color="#888", label="○ = naine (log g > 4)"),
    mpatches.Patch(color="#888", label="▲ = géante (log g < 3)"),
    mpatches.Patch(color="#FFD700", label="★ = point prédit"),
    mpatches.Patch(color="#FFD700", label="- - → flèche vers réel"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="best",
          framealpha=0.85, facecolor="#1A1A2E", labelcolor="white")

ax.set_xlabel("AE axe 1", fontsize=11)
ax.set_ylabel("AE axe 2", fontsize=11)
ax.set_title(
    'Arithmetic latente AE — analogie "word2vec stellaire"\n'
    '★ = point prédit par arithmétique | ▲○ = centroïdes réels',
    fontsize=12, fontweight="bold",
)
ax.grid(True, alpha=0.15)
plt.tight_layout()
arith_path = FIGURES_DIR / "ae_latent_arithmetic.png"
fig.savefig(arith_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"✓ Sauvegardée : {arith_path}")

<div style="
    background: linear-gradient(135deg, rgba(61,64,91,0.10), rgba(44,110,143,0.10));
    border-left: 6px solid #3D405B;
    border-radius: 10px;
    padding: 12px 14px;
    margin: 6px 0 12px 0;
">
<h4 style="margin: 0 0 8px 0; color: #3D405B;">Interprétation — Disentanglement partiel de l'espace latent</h4>
<p style="margin: 0 0 6px 0;">Si l'analogie fonctionne (distance prédite/réelle &lt; 30% de la distance de référence), cela démontre que l'AE a découvert des <strong>directions interprétables</strong> dans l'espace latent : un vecteur "température" (K→G→F) et un vecteur "gravité" (naine→géante) relativement orthogonaux.</p>
<p style="margin: 0 0 6px 0;">Si l'analogie est partielle (30–60%), l'espace latent encode bien ces propriétés mais pas de façon parfaitement linéaire — les deux axes du plan 2D ne sont pas des axes "purs" de température et gravité.</p>
<p style="margin: 0;">Si elle échoue (&gt;60%), cela confirme que l'AE z=2 a trop peu de dimensions pour disentangler complètement T_eff et log g — un argument en faveur du modèle z=3 (§2.bis).</p>
</div>

In [ ]:
# ── Interpolation dans l'espace latent ────────────────────────────────────
Z_interp, X_interp = ae.latent_interpolation(
    X[idx_cold], X[idx_hot],
    n_steps=15,
)

fig, axes = viz.plot_latent_interpolation(
    Z_interp, X_interp,
    feature_names=feature_names,
    label_a=label_a,
    label_b=label_b,
    save_path=FIGURES_DIR / "ae_latent_interpolation.png",
)
plt.show()

print("\nTrajectoire latente :")
print(f"  Départ  : z = ({Z_interp[0,0]:.3f}, {Z_interp[0,1]:.3f})")
print(f"  Arrivée : z = ({Z_interp[-1,0]:.3f}, {Z_interp[-1,1]:.3f})")
print(f"  Distance euclidienne dans l'espace latent : "
      f"{np.linalg.norm(Z_interp[-1] - Z_interp[0]):.3f}")

<a id="anim"></a>

<div style="background: linear-gradient(135deg, #5A2A5E 0%, #2C6E8F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(90,42,94,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">5.bis · Interpolation latente animée — la physique stellaire en mouvement</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(90,42,94,0.10), rgba(44,110,143,0.10));
    border-left: 6px solid #5A2A5E;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<p style="margin: 0 0 8px 0;">La continuité de l'espace latent — vérifiée statiquement en §5 — devient ici <em>visible</em>. En décodant chaque point intermédiaire du chemin linéaire entre une étoile froide K (4200 K) et une étoile chaude F (7984 K), on observe les features spectrales évoluer de façon physiquement cohérente.</p>
<p style="margin: 0 0 8px 0;"><strong>Attendu :</strong> raies de Balmer (Hα, Hβ...) croissantes vers les types chauds, Ca II H&amp;K décroissants, pente du continuum bleuissante, bandes TiO disparaissant.</p>
<p style="margin: 0;"><strong>Ce qui est remarquable :</strong> ce chemin n'a jamais été enseigné au modèle. Il émerge spontanément de la géométrie latente apprise sur 43 019 spectres — c'est la séquence de Harvard reconstruite sans supervision.</p>
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Interpolation latente animée — 3 panneaux simultanés ─────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import warnings

# ── Paramètres de l'animation ─────────────────────────────────────────────
N_FRAMES   = 60          # nombre de frames
FPS_GIF    = 15          # frames/seconde pour le GIF
ALPHAS     = np.linspace(0, 1, N_FRAMES)

# Points de départ et d'arrivée (déjà calculés en §5)
z_cold = Z_ae[idx_cold]   # Z_ae et idx_cold/idx_hot définis en cellule §5
z_hot  = Z_ae[idx_hot]
teff_cold = meta["teff_gspphot"].iloc[idx_cold]
teff_hot  = meta["teff_gspphot"].iloc[idx_hot]

# ── Chemin d'interpolation et décodage ────────────────────────────────────
z_path   = np.array([(1 - a) * z_cold + a * z_hot for a in ALPHAS])  # (N_FRAMES, 2)
X_path   = ae.decode(z_path)                                          # (N_FRAMES, 183) standardisé

# Dé-standardiser pour comparaison physique
if hasattr(loader, 'scaler_') and loader.scaler_ is not None:
    X_path_orig = loader.scaler_.inverse_transform(X_path)
else:
    X_path_orig = X_path.copy()

# ── Familles de features à afficher ──────────────────────────────────────
KEY_FEATURES = [
    # (label_court, nom_feature, couleur, famille)
    ("Hα EW",       "feature_Hα_eq_width",     "#E8593C", "Balmer"),
    ("Hβ EW",       "feature_Hβ_eq_width",     "#E8593C", "Balmer"),
    ("Hδ EW",       "feature_Hdelta_eq_width",  "#F5A623", "Balmer"),
    ("Ca II K EW",  "feature_CaIIK_eq_width",   "#3B8BD4", "Calcium"),
    ("Ca II H EW",  "feature_CaII_H_eq_width",  "#3B8BD4", "Calcium"),
    ("Mg b EW",     "feature_Mg_b_eq_width",    "#4C9B6F", "Métaux"),
    ("FeH proxy",   "feature_FeH_proxy",         "#B07DB8", "Métaux"),
    ("BV synthét.", "feature_synthetic_BV",      "#7F8FA6", "Continuum"),
]

# Valider que les features existent dans feature_names
KEY_FEATURES_OK = [
    (label, col, color, fam)
    for label, col, color, fam in KEY_FEATURES
    if col in feature_names
]
missing = [col for _, col, _, _ in KEY_FEATURES if col not in feature_names]
if missing:
    print(f"⚠  Features absentes (ignorées) : {missing}")

feat_indices = [feature_names.index(col) for _, col, _, _ in KEY_FEATURES_OK]
feat_labels  = [label for label, _, _, _ in KEY_FEATURES_OK]
feat_colors  = [color for _, _, color, _ in KEY_FEATURES_OK]
feat_fams    = [fam   for _, _, _, fam   in KEY_FEATURES_OK]

# Valeurs standardisées du chemin pour les features clés
X_path_keys = X_path[:, feat_indices]  # (N_FRAMES, n_keys) — standardisé

# ── Coloration de l'espace latent par T_eff ───────────────────────────────
teff_vals = meta["teff_gspphot"].values.astype(float)
valid_teff = np.isfinite(teff_vals)

# ── Construction de la figure animée ─────────────────────────────────────
fig = plt.figure(figsize=(16, 6), dpi=120)
gs  = gridspec.GridSpec(1, 3, width_ratios=[1.3, 1.4, 1.3], wspace=0.35)

ax_lat  = fig.add_subplot(gs[0])   # espace latent
ax_feat = fig.add_subplot(gs[1])   # profil features
ax_hr   = fig.add_subplot(gs[2])   # diagramme HR

# ─ Panneau 1 : espace latent ──────────────────────────────────────────────
sc = ax_lat.scatter(
    Z_ae[valid_teff, 0], Z_ae[valid_teff, 1],
    c=teff_vals[valid_teff], cmap="plasma",
    s=0.4, alpha=0.25, rasterized=True,
)
ax_lat.plot(z_path[:, 0], z_path[:, 1], "w-", lw=1.5, alpha=0.85, zorder=3)
dot_lat, = ax_lat.plot([], [], "o", color="white",
                        ms=9, mec="#E8593C", mew=2, zorder=4)
plt.colorbar(sc, ax=ax_lat, label="T_eff (K)", fraction=0.03, pad=0.02)
ax_lat.set_xlabel("AE axe 1", fontsize=9)
ax_lat.set_ylabel("AE axe 2", fontsize=9)
ax_lat.set_title("Espace latent AE z=2", fontsize=10, fontweight="bold")
title_lat = ax_lat.text(
    0.5, -0.16, "",
    transform=ax_lat.transAxes, ha="center", fontsize=9, color="#2A3B4C"
)

# ─ Panneau 2 : profil features ────────────────────────────────────────────
n_keys = len(feat_labels)
y_pos  = np.arange(n_keys)

bars = ax_feat.barh(
    y_pos,
    np.zeros(n_keys),
    color=feat_colors, height=0.65,
    edgecolor="white", linewidth=0.5,
)
ax_feat.set_yticks(y_pos)
ax_feat.set_yticklabels(feat_labels, fontsize=8)
ax_feat.set_xlim(-3, 3)
ax_feat.axvline(0, color="gray", lw=0.8, ls="--", alpha=0.6)
ax_feat.set_xlabel("Score standardisé (σ)", fontsize=9)
ax_feat.grid(axis="x", alpha=0.2)

# Légende familles
family_handles = [
    Line2D([0], [0], color=c, lw=4, label=f)
    for f, c in zip(["Balmer", "Calcium", "Métaux", "Continuum"],
                    ["#E8593C", "#3B8BD4", "#4C9B6F", "#7F8FA6"])
    if f in feat_fams
]
ax_feat.legend(handles=family_handles, fontsize=7,
               loc="lower right", framealpha=0.85)
title_feat = ax_feat.set_title("", fontsize=10, fontweight="bold")

# ─ Panneau 3 : diagramme HR ───────────────────────────────────────────────
logg_vals = meta["logg_gspphot"].values.astype(float)
valid_hr  = valid_teff & np.isfinite(logg_vals)
ax_hr.scatter(
    teff_vals[valid_hr], logg_vals[valid_hr],
    c=teff_vals[valid_hr], cmap="plasma",
    s=0.4, alpha=0.2, rasterized=True,
)
ax_hr.set_xlim(9000, 2500)
ax_hr.set_ylim(5.5, 0.0)
ax_hr.set_xlabel("T_eff Gaia (K)", fontsize=9)
ax_hr.set_ylabel("log g Gaia", fontsize=9)
ax_hr.set_title("Diagramme HR", fontsize=10, fontweight="bold")
dot_hr, = ax_hr.plot([], [], "o", color="white",
                      ms=9, mec="#E8593C", mew=2, zorder=4)

# Corrélation AE axe 2 → T_eff proxy pour positionner sur HR
# On utilise un mapping linéaire z_hot→T_eff_hot, z_cold→T_eff_cold
logg_cold = float(meta["logg_gspphot"].iloc[idx_cold])
logg_hot  = float(meta["logg_gspphot"].iloc[idx_hot])

fig.suptitle(
    "Interpolation dans l'espace latent AE — la séquence de Harvard en mouvement\n"
    f"De {teff_cold:.0f} K (étoile K) à {teff_hot:.0f} K (étoile F) · {N_FRAMES} étapes",
    fontsize=11, fontweight="bold", y=1.02,
)

# ── Fonction de mise à jour par frame ────────────────────────────────────
def update(frame):
    a = ALPHAS[frame]
    teff_est  = (1 - a) * teff_cold + a * teff_hot
    logg_est  = (1 - a) * logg_cold + a * logg_hot

    # Panneau 1 : point dans l'espace latent
    dot_lat.set_data([z_path[frame, 0]], [z_path[frame, 1]])
    title_lat.set_text(f"T_eff ≈ {teff_est:.0f} K")

    # Panneau 2 : barres features
    vals = X_path_keys[frame]
    for bar, v in zip(bars, vals):
        bar.set_width(v)
    title_feat.set_text(f"Étape {frame + 1}/{N_FRAMES}  —  T_eff ≈ {teff_est:.0f} K")

    # Panneau 3 : point sur HR
    dot_hr.set_data([teff_est], [logg_est])

    return [dot_lat, title_lat, *bars, title_feat, dot_hr]

# ── Rendu ────────────────────────────────────────────────────────────────
ani = animation.FuncAnimation(
    fig, update, frames=N_FRAMES,
    interval=1000 // FPS_GIF,
    blit=True, repeat=True,
)

gif_path = FIGURES_DIR / "ae_latent_animation.gif"
print(f"Génération du GIF ({N_FRAMES} frames, {FPS_GIF} fps)...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ani.save(str(gif_path), writer="pillow", fps=FPS_GIF, dpi=100)
print(f"✓ GIF sauvegardé : {gif_path}")

plt.tight_layout()
plt.show()

### 5.bis.2 · Version interactive — Plotly slider

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.12), rgba(90,42,94,0.12));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 12px 14px;
    margin-bottom: 10px;
">
La version Plotly produit un HTML interactif : un <strong>slider</strong> permet de naviguer manuellement à travers les 60 étapes de l'interpolation. Idéal pour la présentation au Musée de la Civilisation — on peut s'arrêter à mi-chemin, zoomer, et interagir avec le public.
</div>

In [ ]:
# ── Interpolation latente — version Plotly slider interactif ─────────────
try:
    import plotly.graph_objects as go
    _PLOTLY_ANIM = True
except ImportError:
    print("⚠  Plotly non installé → pip install plotly")
    _PLOTLY_ANIM = False

if _PLOTLY_ANIM:
    import pandas as pd
    from scipy.stats import spearmanr

    # ── Données Plotly ────────────────────────────────────────────────────
    # Sous-échantillonnage pour réactivité
    N_BG = min(15_000, len(Z_ae))
    rng_plt = np.random.default_rng(42)
    idx_bg  = rng_plt.choice(len(Z_ae), N_BG, replace=False)

    teff_bg = meta["teff_gspphot"].values[idx_bg]
    valid_bg = np.isfinite(teff_bg)
    Z_bg     = Z_ae[idx_bg][valid_bg]
    T_bg     = teff_bg[valid_bg]

    # Normalisation T_eff pour colorscale
    t_min, t_max = float(np.percentile(T_bg, 2)), float(np.percentile(T_bg, 98))

    # Chemin de décodage (score standardisé → valeurs pour chaque frame)
    # On prépare les features clés pour chaque frame
    frames_data = []
    for fi in range(N_FRAMES):
        a = ALPHAS[fi]
        teff_est = (1 - a) * float(teff_cold) + a * float(teff_hot)
        vals = X_path_keys[fi].tolist()
        frames_data.append((fi, teff_est, vals))

    # ── Layout de base ────────────────────────────────────────────────────
    fig_slider = go.Figure()

    # Trace background (espace latent, coloré par T_eff)
    fig_slider.add_trace(go.Scatter(
        x=Z_bg[:, 0].tolist(), y=Z_bg[:, 1].tolist(),
        mode="markers",
        marker=dict(
            size=2, opacity=0.3,
            color=T_bg.tolist(),
            colorscale="Plasma",
            cmin=t_min, cmax=t_max,
            colorbar=dict(title="T_eff (K)", x=0.48),
        ),
        name="Étoiles (fond)",
        hoverinfo="skip",
    ))

    # Trace chemin d'interpolation
    fig_slider.add_trace(go.Scatter(
        x=z_path[:, 0].tolist(), y=z_path[:, 1].tolist(),
        mode="lines",
        line=dict(color="white", width=2, dash="dot"),
        name="Chemin interpolation",
        hoverinfo="skip",
    ))

    # Trace point mobile (frame 0)
    fig_slider.add_trace(go.Scatter(
        x=[z_path[0, 0]], y=[z_path[0, 1]],
        mode="markers",
        marker=dict(size=14, color="#E8593C",
                    line=dict(color="white", width=2)),
        name="Point courant",
    ))

    # Trace barres features (frame 0) — rendu comme barh via scatter horizontal
    vals0 = X_path_keys[0]
    fig_slider.add_trace(go.Bar(
        x=vals0.tolist(),
        y=feat_labels,
        orientation="h",
        marker_color=feat_colors,
        name="Features spectrales",
        xaxis="x2", yaxis="y2",
        showlegend=False,
    ))

    # ── Frames Plotly ─────────────────────────────────────────────────────
    plotly_frames = []
    for fi, teff_est, vals in frames_data:
        frame = go.Frame(
            data=[
                go.Scatter(),                                    # bg (inchangé)
                go.Scatter(),                                    # chemin (inchangé)
                go.Scatter(
                    x=[z_path[fi, 0]], y=[z_path[fi, 1]],
                ),
                go.Bar(
                    x=vals,
                    y=feat_labels,
                ),
            ],
            name=str(fi),
            layout=go.Layout(
                title_text=(
                    f"Interpolation latente AE — Étape {fi+1}/{N_FRAMES} · "
                    f"T_eff ≈ {teff_est:.0f} K"
                )
            ),
        )
        plotly_frames.append(frame)
    fig_slider.frames = plotly_frames

    # ── Slider et boutons ─────────────────────────────────────────────────
    sliders = [dict(
        active=0,
        currentvalue=dict(prefix="Étape : ", visible=True, xanchor="center"),
        pad=dict(b=10, t=50),
        steps=[
            dict(
                args=[[str(fi)],
                      dict(frame=dict(duration=0, redraw=True),
                           mode="immediate")],
                label=str(fi + 1),
                method="animate",
            )
            for fi in range(N_FRAMES)
        ],
    )]

    play_button = dict(
        label="▶ Play",
        method="animate",
        args=[None, dict(
            frame=dict(duration=80, redraw=True),
            fromcurrent=True,
            mode="immediate",
        )],
    )
    pause_button = dict(
        label="⏸ Pause",
        method="animate",
        args=[[None], dict(frame=dict(duration=0, redraw=False),
                          mode="immediate")],
    )

    fig_slider.update_layout(
        title=(
            f"Interpolation latente AE · Étape 1/{N_FRAMES} · "
            f"T_eff ≈ {float(teff_cold):.0f} K"
        ),
        plot_bgcolor="#1A1A2E",
        paper_bgcolor="#16213E",
        font=dict(color="white", size=11),
        xaxis=dict(
            domain=[0, 0.45],
            title="AE axe 1",
            gridcolor="#333",
        ),
        yaxis=dict(
            title="AE axe 2",
            gridcolor="#333",
        ),
        xaxis2=dict(
            domain=[0.55, 1.0],
            title="Score standardisé (σ)",
            range=[-3, 3],
            gridcolor="#333",
            zeroline=True, zerolinecolor="white", zerolinewidth=1,
        ),
        yaxis2=dict(
            anchor="x2",
            gridcolor="#333",
        ),
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            x=0.5, y=1.08,
            xanchor="center",
            buttons=[play_button, pause_button],
        )],
        sliders=sliders,
        height=600,
        width=1100,
        margin=dict(l=60, r=40, t=100, b=80),
        showlegend=False,
    )

    # ── Sauvegarde HTML ───────────────────────────────────────────────────
    slider_path = FIGURES_DIR / "ae_interpolation_slider.html"
    fig_slider.write_html(str(slider_path), include_plotlyjs="cdn")
    print(f"✓ Slider interactif sauvegardé : {slider_path}")
    print("  → Ouvrir dans un navigateur pour la présentation au Musée")
    fig_slider.show()

<div style="
    background: linear-gradient(135deg, rgba(90,42,94,0.10), rgba(44,110,143,0.10));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 10px 0; color: #2C6E8F;">Interprétation — La séquence de Harvard reconstruite</h4>
<p style="margin: 0 0 8px 0;">Si l'animation est physiquement cohérente, on devrait observer :</p>
<ul style="margin: 0 0 8px 0; padding-left: 18px; line-height: 1.6;">
  <li><strong>Hα, Hβ, Hδ</strong> croissants vers les types chauds (maximum vers A, F)</li>
  <li><strong>Ca II K &amp; H</strong> décroissants (absorbants dans les étoiles froides K, faibles dans A)</li>
  <li><strong>Mg b, FeH proxy</strong> décroissants (métallicité apparente liée à la gravité et la température)</li>
  <li><strong>BV synthétique</strong> décroissant (spectre se bleuit vers les étoiles chaudes)</li>
</ul>
<p style="margin: 0;">Tout écart à ces tendances signale soit une non-linéarité de l'AE (l'espace latent n'est pas parfaitement organisé selon la physique), soit une corrélation inattendue entre paramètres physiques — les deux sont scientifiquement intéressants.</p>
</div>

<a id="synthesis"></a>

<div style="background: linear-gradient(135deg, #5A2A5E 0%, #2C6E8F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(90,42,94,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">6 · Synthèse comparative PCA / UMAP / Autoencodeur</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
La figure de synthèse 3x3 et le tableau quantitatif rassemblent les compromis entre interprétabilité, non-linéarité et reconstruction.
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
<strong>6.0 Chargement des embeddings de référence (NB02)</strong>
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Chargement des embeddings UMAP/t-SNE (sauvegardés par NB02) ──────────
# NB02 sauvegarde dans phy3500_embeddings.joblib — PAS dans Z_umap.npy.
# Bug corrigé : l'ancien code cherchait un fichier .npy inexistant.

embeddings_path = Path(paths["REPORTS_DIR"]) / "phy3500_embeddings.joblib"

if embeddings_path.exists():
    embeddings_output = joblib.load(embeddings_path)
    Z_umap = embeddings_output["Z_umap"]
    Z_tsne = embeddings_output.get("Z_tsne", None)
    print(f"✓ Embeddings chargés depuis {embeddings_path.name}")
    print(f"  Z_umap : {Z_umap.shape}")
    if Z_tsne is not None:
        print(f"  Z_tsne : {Z_tsne.shape}")
else:
    Z_umap = None
    Z_tsne = None
    print("⚠  Fichier phy3500_embeddings.joblib introuvable.")
    print("   → Lancer d'abord 02_umap_tsne.ipynb (cellule 7 — Sauvegarde).")

<a id="inference"></a>

<div style="background: linear-gradient(135deg, #2C6E8F 0%, #5A2A5E 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(44,110,143,0.30);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">6.bis · Inférence — tester un nouveau candidat</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.12), rgba(90,42,94,0.12));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<p style="margin: 0 0 8px 0;">C'est ici que l'autoencodeur prouve sa <strong>supériorité sur t-SNE</strong> : il est paramétrique, il a appris une <em>fonction mathématique</em>, et peut donc projeter n'importe quel nouveau spectre sans recalcul. Un nouveau candidat passe par le pipeline en quelques millisecondes.</p>
<p style="margin: 0 0 8px 0;">La fonction <code>tester_candidat()</code> produit :</p>
<ol style="margin: 0; padding-left: 18px; line-height: 1.6;">
  <li>Les <strong>coordonnées latentes</strong> (z₁, z₂) → position sur la carte</li>
  <li>Le <strong>cluster HDBSCAN le plus proche</strong> → héritage des propriétés physiques</li>
  <li>La <strong>MSE de reconstruction</strong> → score d'anomalie avec verdict</li>
  <li>Une <strong>figure 3 panneaux</strong> : position sur l'embedding + profil features + jauge MSE</li>
</ol>
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #2C6E8F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── tester_candidat() — inférence complète sur un nouveau spectre ─────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

def tester_candidat(
    x_raw,
    ae_model,
    loader_ref,
    feature_names_ref,
    Z_train,
    cluster_labels_train=None,
    meta_train=None,
    y_train=None,
    teff_train=None,
    label="Candidat",
    save_path=None,
):
    """
    Projette un nouveau spectre dans l'espace latent AE et produit une
    fiche diagnostique complète.

    Parameters
    ----------
    x_raw : np.ndarray (1, n_features) ou (n_features,)
        Features BRUTES (non standardisées) du candidat.
        Doivent être dans le même ordre que feature_names_ref.
    ae_model : SpectralAutoencoder
        Modèle entraîné (z=2).
    loader_ref : DimRedDataLoader
        Loader ayant servi à l'entraînement (contient le scaler).
    feature_names_ref : list[str]
        Liste des 183 noms de features dans l'ordre du modèle.
    Z_train : np.ndarray (N, 2)
        Espace latent du dataset d'entraînement (pour le scatter).
    cluster_labels_train : np.ndarray (N,), optionnel
        Labels HDBSCAN sur le dataset d'entraînement.
    Returns
    -------
    dict : z (coordonnées latentes), mse, cluster, verdict
    """
    x = np.atleast_2d(x_raw).astype(float)  # (1, D)

    # 1. Standardisation avec le scaler du loader
    if hasattr(loader_ref, 'scaler_') and loader_ref.scaler_ is not None:
        x_scaled = loader_ref.scaler_.transform(x)
    else:
        x_scaled = x.copy()

    # 2. Encodage → coordonnées latentes
    z = ae_model.encode(x_scaled)  # (1, 2)

    # 3. Reconstruction
    x_recon_scaled = ae_model.reconstruct(x_scaled)
    mse = float(np.mean((x_scaled - x_recon_scaled) ** 2))

    # 4. Cluster le plus proche (k-NN sur l'espace latent)
    dists = np.linalg.norm(Z_train - z, axis=1)
    nn_idx = np.argsort(dists)[:50]  # 50 plus proches voisins
    if cluster_labels_train is not None:
        nn_clusters = cluster_labels_train[nn_idx]
        nn_clusters_valid = nn_clusters[nn_clusters >= 0]
        if len(nn_clusters_valid) > 0:
            cluster_pred = int(np.bincount(nn_clusters_valid).argmax())
        else:
            cluster_pred = -1
    else:
        cluster_pred = None

    # 5. Verdict anomalie basé sur MSE
    mse_stars = float(np.mean((Z_train - Z_train.mean(axis=0))**2))
    # Seuils basés sur la distribution des MSE d'entraînement
    mse_q95 = 0.97  # depuis le rapport (STAR q95 = 0.974)
    mse_q99 = 3.0   # estimation
    if mse < mse_q95:
        verdict = "🟢 BANAL — spectre typique du dataset"
    elif mse < mse_q99:
        verdict = "🟡 INHABITUEL — dans le top 5% des erreurs"
    else:
        verdict = "🔴 ANOMALIE — objet très atypique"

    # ── Figure 3 panneaux ────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 6), dpi=150)
    gs  = gridspec.GridSpec(1, 3, width_ratios=[1.4, 1.3, 1.3], wspace=0.35)
    ax1 = fig.add_subplot(gs[0])  # Espace latent
    ax2 = fig.add_subplot(gs[1])  # Profil features
    ax3 = fig.add_subplot(gs[2])  # Jauge MSE

    # Panneau 1 : espace latent
    if teff_train is not None:
        valid_t = np.isfinite(teff_train)
        sc = ax1.scatter(
            Z_train[valid_t, 0], Z_train[valid_t, 1],
            c=teff_train[valid_t], cmap="plasma",
            s=0.5, alpha=0.2, rasterized=True, zorder=1,
        )
        plt.colorbar(sc, ax=ax1, label="T_eff (K)", fraction=0.03)
    else:
        ax1.scatter(Z_train[:, 0], Z_train[:, 1],
                    c="#4A4A4A", s=0.5, alpha=0.2, rasterized=True)

    # Point candidat
    ax1.scatter(z[0, 0], z[0, 1], s=300, c="#FFD700", marker="*",
                edgecolors="white", linewidths=1.5, zorder=6)
    ax1.annotate(
        f"{label}\n({z[0,0]:.3f}, {z[0,1]:.3f})",
        (z[0, 0], z[0, 1]),
        textcoords="offset points", xytext=(10, 8),
        fontsize=9, fontweight="bold", color="#FFD700",
        bbox=dict(boxstyle="round,pad=0.3", fc="black", alpha=0.75),
        zorder=7,
    )
    if cluster_pred is not None and cluster_pred >= 0:
        ax1.set_title(
            f"Position latente — Cluster C{cluster_pred}",
            fontsize=11, fontweight="bold",
        )
    else:
        ax1.set_title("Position latente — Bruit/outlier", fontsize=11)
    ax1.set_xlabel("AE axe 1"); ax1.set_ylabel("AE axe 2")
    ax1.grid(True, alpha=0.15)

    # Panneau 2 : profil features clés
    _KEY_F_LABELS = [
        "Hα EW", "Hβ EW", "Ca II K", "Mg b", "FeH proxy", "BV synthét."
    ]
    _KEY_F_NAMES = [
        "feature_Hα_eq_width", "feature_Hβ_eq_width",
        "feature_CaIIK_eq_width", "feature_Mg_b_eq_width",
        "feature_FeH_proxy", "feature_synthetic_BV",
    ]
    _KEY_F_COLS = ["#E8593C", "#E8593C", "#3B8BD4", "#4C9B6F", "#B07DB8", "#7F8FA6"]
    _valid_kf = [(l, n, c) for l, n, c in zip(_KEY_F_LABELS, _KEY_F_NAMES, _KEY_F_COLS)
                 if n in feature_names_ref]
    kf_vals_orig  = [x_scaled[0, feature_names_ref.index(n)] for _, n, _ in _valid_kf]
    kf_vals_recon = [x_recon_scaled[0, feature_names_ref.index(n)] for _, n, _ in _valid_kf]
    kf_labels     = [l for l, _, _ in _valid_kf]
    kf_colors     = [c for _, _, c in _valid_kf]
    y_kf = np.arange(len(kf_labels))
    w_kf = 0.35
    ax2.barh(y_kf - w_kf/2, kf_vals_orig,  w_kf, color=kf_colors, alpha=0.90,
             edgecolor="white", linewidth=0.4, label="Original")
    ax2.barh(y_kf + w_kf/2, kf_vals_recon, w_kf, color=kf_colors, alpha=0.35,
             edgecolor=kf_colors, linewidth=1.2, label="Reconstruit")
    ax2.set_yticks(y_kf)
    ax2.set_yticklabels(kf_labels, fontsize=9)
    ax2.axvline(0, color="gray", lw=0.7, ls="--")
    ax2.set_xlabel("Score standardisé (σ)", fontsize=9)
    ax2.set_title("Profil spectroscopique clé", fontsize=11, fontweight="bold")
    ax2.legend(fontsize=8)
    ax2.grid(axis="x", alpha=0.2)

    # Panneau 3 : jauge MSE
    mse_display = min(mse, 10.0)  # Clamp pour affichage
    mse_norm = mse_display / 10.0
    cmap_gauge = plt.cm.RdYlGn_r
    for j in range(100):
        ax3.barh(0, 0.1, left=j * 0.1, height=0.4,
                 color=cmap_gauge(j / 100), alpha=0.7)
    ax3.barh(0, mse_norm * 10, height=0.4, color="none",
             edgecolor="black", linewidth=2)
    ax3.axvline(mse_display, color="black", lw=3, ymin=0.1, ymax=0.9)
    ax3.axvline(mse_q95, color="orange", lw=1.5, ls="--",
                label=f"q95 = {mse_q95:.2f}")
    ax3.axvline(mse_q99, color="red", lw=1.5, ls="--",
                label=f"q99 ≈ {mse_q99:.2f}")
    ax3.set_xlim(0, 10)
    ax3.set_ylim(-0.5, 1)
    ax3.set_xlabel("MSE de reconstruction", fontsize=10)
    ax3.set_yticks([])
    ax3.set_title(
        f"Score d'anomalie\nMSE = {mse:.4f}",
        fontsize=11, fontweight="bold",
    )
    ax3.text(0.5, 0.75, verdict, transform=ax3.transAxes,
             ha="center", va="center", fontsize=10, fontweight="bold",
             bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#AAA", alpha=0.9))
    ax3.legend(fontsize=8, loc="upper right")

    fig.suptitle(
        f"Fiche diagnostique AE — {label}",
        fontsize=13, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
        print(f"✓ Sauvegardée : {save_path}")
    plt.show()

    return {
        "z":           z[0],
        "mse":         mse,
        "cluster":     cluster_pred,
        "verdict":     verdict,
        "x_scaled":    x_scaled[0],
        "x_recon":     x_recon_scaled[0],
    }


# ── Démonstration : 3 candidats de types différents ──────────────────────
print("Démonstration de tester_candidat() sur 3 spectres réels du dataset\n")
print("(on utilise des spectres du dataset en guise de 'nouveaux candidats')")
print("-" * 65)

# Récupérer les clusters depuis NB02 si disponibles
_cluster_avail = "cluster_labels" in dir() and cluster_labels is not None
_clusters_for_candidat = cluster_labels if _cluster_avail else None

# Récupérer T_eff pour le fond coloré
_teff_for_candidat = meta["teff_gspphot"].values.astype(float)

# Dé-standardisation pour récupérer x_raw
if hasattr(loader, 'scaler_') and loader.scaler_ is not None:
    X_raw_all = loader.scaler_.inverse_transform(X)
else:
    X_raw_all = X.copy()

# Sélection de 3 candidats représentatifs
teff_all = meta["teff_gspphot"].values.astype(float)
logg_all = meta["logg_gspphot"].values.astype(float)

_CANDIDATS = [
    # (label, critère T_eff, critère log g, couleur)
    ("Étoile A typique",  (7500, 10000), (3.8, 5.5)),
    ("Étoile G naine",    (5500, 6000),  (4.2, 4.8)),
    ("Étoile K géante",   (4000, 4800),  (1.0, 2.5)),
]

rng_c = np.random.default_rng(99)
results_candidats = []

for label_c, (t_lo, t_hi), (g_lo, g_hi) in _CANDIDATS:
    mask_c = (
        np.isfinite(teff_all) & np.isfinite(logg_all)
        & (teff_all >= t_lo) & (teff_all < t_hi)
        & (logg_all >= g_lo) & (logg_all < g_hi)
        & (y == "STAR")
    )
    candidates = np.where(mask_c)[0]
    if len(candidates) == 0:
        print(f"  ⚠  Aucun spectre trouvé pour : {label_c}")
        continue
    idx_c = int(rng_c.choice(candidates))
    teff_c = teff_all[idx_c]
    logg_c = logg_all[idx_c]
    print(f"\n{'='*65}")
    print(f"  Candidat : {label_c}")
    print(f"  T_eff Gaia = {teff_c:.0f} K | log g = {logg_c:.2f}")
    print(f"  obsid index = {idx_c}")

    res = tester_candidat(
        x_raw=X_raw_all[idx_c:idx_c+1],
        ae_model=ae,
        loader_ref=loader,
        feature_names_ref=feature_names,
        Z_train=Z_ae,
        cluster_labels_train=_clusters_for_candidat,
        teff_train=_teff_for_candidat,
        label=label_c,
        save_path=FIGURES_DIR / f"ae_candidat_{label_c.replace(' ', '_')}.png",
    )
    results_candidats.append((label_c, res))
    print(f"  z = ({res['z'][0]:.4f}, {res['z'][1]:.4f})")
    print(f"  MSE = {res['mse']:.5f}")
    print(f"  Verdict : {res['verdict']}")

print("\n" + "="*65)
print("  Récapitulatif")
print("="*65)
for label_c, res in results_candidats:
    cluster_str = f"C{res['cluster']}" if res['cluster'] is not None and res['cluster'] >= 0 else "Bruit"
    print(f"  {label_c:25s} | MSE={res['mse']:.4f} | Cluster={cluster_str} | {res['verdict']}")

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.12), rgba(90,42,94,0.12));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 12px 14px;
    margin: 6px 0 12px 0;
">
<h4 style="margin: 0 0 8px 0; color: #2C6E8F;">Conclusion — L'autoencodeur comme outil d'analyse spectroscopique</h4>
<p style="margin: 0 0 6px 0;">La fonction <code>tester_candidat()</code> peut être appelée sur n'importe quel spectre du futur catalogue LAMOST DR6, ou même sur des spectres d'autres surveys (SDSS, GALAH) après adaptation du pipeline de features. C'est la preuve que l'AE n'a pas <em>mémorisé</em> les données — il a appris une <strong>représentation générale</strong> de la physique spectrale stellaire.</p>
<p style="margin: 0;">Pour un usage en production, on chargerait <code>ae_latent2.pt</code> et <code>pca_analyzer.joblib</code>, et le pipeline complet (features → StandardScaler → encode → cluster assignment + MSE) s'exécute en &lt;1ms par spectre — adapté à des catalogues de millions d'objets.</p>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 8px 0;
">
<strong>6.1 Figure de synthèse 3x3 - PCA / UMAP / Autoencodeur</strong><br>
Grille comparative : lignes = méthodes, colonnes = colorations physiques.
</div>

In [ ]:
# ── Figure de synthèse 3×3 : PCA / UMAP / Autoencodeur ──────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

# Palette classes LAMOST (top-level)
CLASS_PALETTE = {
    "STAR":    "#4C72B0",
    "GALAXY":  "#DD8452",
    "QSO":     "#55A868",
    "UNKNOWN": "#8172B3",
}
CLASS_MARKER = {
    "STAR": "o", "GALAXY": "s", "QSO": "^", "UNKNOWN": "x",
}

# Méthodes disponibles (Z_umap peut être None si NB02 non exécuté)
METHODS = []
METHODS.append(("PCA",          scores_pca[:, :2], "PC1",      "PC2"))
if Z_umap is not None:
    METHODS.append(("UMAP",      Z_umap,            "UMAP 1",   "UMAP 2"))
METHODS.append(("Autoencodeur", Z_ae,              "Latent 1", "Latent 2"))

PARAMS = [
    ("Classe spectrale", None,              None),
    ("T_eff (K)",        "teff_gspphot",   "plasma"),
    ("[Fe/H]",           "mh_gspphot",     "RdYlBu_r"),
]

n_rows = len(METHODS)
n_cols = len(PARAMS)
fig = plt.figure(figsize=(5.5 * n_cols, 4.5 * n_rows), dpi=150)
gs  = gridspec.GridSpec(
    n_rows, n_cols, figure=fig,
    hspace=0.45, wspace=0.35,
    left=0.07, right=0.96, top=0.91, bottom=0.06,
)

for row, (method_name, Z_m, xlabel, ylabel) in enumerate(METHODS):
    for col, (param_label, param_col, cmap_name) in enumerate(PARAMS):
        ax = fig.add_subplot(gs[row, col])

        if param_col is None:
            # ── Coloration par classe ─────────────────────────────────────
            for cls in np.unique(y):
                mask = y == cls
                ax.scatter(
                    Z_m[mask, 0], Z_m[mask, 1],
                    c=CLASS_PALETTE.get(cls, "#888888"),
                    marker=CLASS_MARKER.get(cls, "o"),
                    s=2, alpha=0.45, linewidths=0,
                    label=cls, rasterized=True,
                )
            # Légende uniquement sur le premier panneau
            if row == 0:
                ax.legend(
                    markerscale=4, fontsize=8, framealpha=0.8,
                    loc="best", handletextpad=0.3,
                )
        else:
            # ── Coloration par paramètre continu ──────────────────────────
            if param_col in meta.columns:
                vals = meta[param_col].values.astype(float)
                valid = np.isfinite(vals)
                # Points sans paramètre Gaia en gris clair
                if (~valid).any():
                    ax.scatter(
                        Z_m[~valid, 0], Z_m[~valid, 1],
                        c="#dddddd", s=1, alpha=0.15,
                        rasterized=True, linewidths=0,
                    )
                sc = ax.scatter(
                    Z_m[valid, 0], Z_m[valid, 1],
                    c=vals[valid],
                    cmap=cmap_name,
                    vmin=np.nanpercentile(vals[valid], 2),
                    vmax=np.nanpercentile(vals[valid], 98),
                    s=2, alpha=0.55, linewidths=0,
                    rasterized=True,
                )
                plt.colorbar(
                    sc, ax=ax,
                    fraction=0.046, pad=0.04,
                    label=param_label,
                ).ax.tick_params(labelsize=7)

        # ── Cosmétique commune ────────────────────────────────────────────
        ax.set_aspect("equal", "box")
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.2, linewidth=0.5)
        ax.set_xlabel(xlabel, fontsize=8)

        # En-têtes : méthode à gauche, paramètre en haut
        if col == 0:
            ax.set_ylabel(
                f"{method_name}\n{ylabel}",
                fontsize=9, fontweight="bold",
            )
        else:
            ax.set_ylabel(ylabel, fontsize=8)
        if row == 0:
            ax.set_title(param_label, fontsize=11, fontweight="bold", pad=6)

fig.suptitle(
    "Synthèse comparative — PCA / UMAP / Autoencodeur\n"
    "Spectres stellaires LAMOST DR5 × Gaia DR3",
    fontsize=14, fontweight="bold",
)

synth_path = FIGURES_DIR / "synthesis_pca_umap_ae.png"
fig.savefig(synth_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"\n✓ Figure de synthèse sauvegardée : {synth_path}")

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 8px 0;
">
<strong>6.2 Tableau comparatif quantitatif</strong><br>
Corrélations de température, erreur de reconstruction et propriétés qualitatives par méthode.
</div>

In [ ]:
# ── Tableau comparatif quantitatif ──────────────────────────────────────
from scipy.stats import spearmanr

teff_vals  = meta["teff_gspphot"].values.astype(float)
teff_valid = np.isfinite(teff_vals)

def _spearman_ax(Z_m, axis=0):
    """Corrélation Spearman entre l'axe `axis` de Z_m et T_eff."""
    if Z_m is None:
        return float('nan')
    r, _ = spearmanr(Z_m[teff_valid, axis], teff_vals[teff_valid])
    return r

results = {}

# PCA
mse_pca2 = float(pca.reconstruction_error(X, n_components=2).mean())
r_pca = _spearman_ax(scores_pca)
results["PCA (PC1/PC2)"] = {
    "ρ(axe 1, T_eff)": f"{r_pca:+.3f}",
    "MSE recon. (2 dims)": f"{mse_pca2:.5f}",
    "Paramétrique": "Oui",
    "Non-linéaire": "Non",
    "Interprétable": "Oui (loadings)",
}

# Autoencodeur
mse_ae = float(ae.reconstruction_mse(X))
r_ae   = _spearman_ax(Z_ae)
results["Autoencodeur (z=2)"] = {
    "ρ(axe 1, T_eff)": f"{r_ae:+.3f}",
    "MSE recon. (2 dims)": f"{mse_ae:.5f}",
    "Paramétrique": "Oui",
    "Non-linéaire": "Oui",
    "Interprétable": "Partielle",
}

# UMAP
if Z_umap is not None:
    r_umap = _spearman_ax(Z_umap)
    results["UMAP"] = {
        "ρ(axe 1, T_eff)": f"{r_umap:+.3f}",
        "MSE recon. (2 dims)": "N/A",
        "Paramétrique": "Oui (partiel)",
        "Non-linéaire": "Oui",
        "Interprétable": "Non",
    }

df_summary = pd.DataFrame(results).T
print("\n" + "=" * 65)
print("  RÉSUMÉ — NB03 : Comparaison PCA / UMAP / Autoencodeur")
print("=" * 65)
print(df_summary.to_string())
print("=" * 65)
print(f"\n  Temps entraînement AE : {ae.fit_time_:.1f} s")
print(f"  Epochs effectuées     : {len(ae.history_['train_loss'])}")
print(f"  Best val_loss         : {min(ae.history_['val_loss']):.6f}")

<a id="save"></a>

<div style="background: linear-gradient(135deg, #2C6E8F 0%, #5A2A5E 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(44,110,143,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">7 · Sauvegarde des sorties et rapports de run</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.10), rgba(90,42,94,0.10));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 14px 16px;
    margin: 0 0 10px 0;
">
Cette section enregistre automatiquement les artefacts du run (joblib, JSON, TXT), avec un fichier horodaté et une version latest.
</div>

### Pourquoi c’est important
- Traçabilité complète des hyperparamètres, métriques et figures.
- Réutilisation directe des artefacts dans les notebooks suivants et le rapport.
- Reproductibilité stricte des résultats expérimentaux.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #2C6E8F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
import json
import joblib
from datetime import datetime, timezone


def _class_counts(labels):
    classes, counts = np.unique(labels, return_counts=True)
    return {str(cls): int(cnt) for cls, cnt in zip(classes, counts)}


def _safe_df_records(df, max_rows=None):
    if df is None:
        return None
    out = df.copy()
    if max_rows is not None:
        out = out.head(max_rows)
    out = out.where(pd.notna(out), None)
    return out.to_dict(orient="records")


def _json_default(obj):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (datetime, pd.Timestamp)):
        return obj.isoformat()
    raise TypeError(f"Type non serialisable: {type(obj)!r}")


def _embedding_stats(Z):
    if Z is None:
        return None
    arr = np.asarray(Z)
    if arr.ndim != 2 or arr.shape[1] < 2:
        return {"shape": list(arr.shape)}
    return {
        "shape": list(arr.shape),
        "x_min": float(np.nanmin(arr[:, 0])),
        "x_max": float(np.nanmax(arr[:, 0])),
        "y_min": float(np.nanmin(arr[:, 1])),
        "y_max": float(np.nanmax(arr[:, 1])),
        "x_mean": float(np.nanmean(arr[:, 0])),
        "y_mean": float(np.nanmean(arr[:, 1])),
        "x_std": float(np.nanstd(arr[:, 0])),
        "y_std": float(np.nanstd(arr[:, 1])),
    }


def _fmt_seconds(value):
    if value is None:
        return "N/A"
    try:
        v = float(value)
    except (TypeError, ValueError):
        return "N/A"
    return f"{v:.2f}s" if np.isfinite(v) else "N/A"


required_vars = ["X", "y", "meta", "ae", "feature_names", "paths", "FIGURES_DIR"]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        "Variables manquantes pour la sauvegarde. "
        "Executer les cellules precedentes du notebook avant cette cellule.\n"
        f"Manquantes: {missing}"
    )


timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = Path(paths["REPORTS_DIR"]) / "runs" / "phy3500_autoencoder"
run_dir.mkdir(parents=True, exist_ok=True)

Z_ae_current = Z_ae if "Z_ae" in globals() else ae.encode(X)
X_recon_current = X_recon if "X_recon" in globals() else None
history_current = history if "history" in globals() else getattr(ae, "history_", None)

train_loss = history_current.get("train_loss", []) if isinstance(history_current, dict) else []
val_loss = history_current.get("val_loss", []) if isinstance(history_current, dict) else []

ae_fit_time = getattr(ae, "fit_time_", None)
epochs_done = len(train_loss) if isinstance(train_loss, list) else None
best_val_loss = float(min(val_loss)) if len(val_loss) > 0 else None
final_train_loss = float(train_loss[-1]) if len(train_loss) > 0 else None
final_val_loss = float(val_loss[-1]) if len(val_loss) > 0 else None

try:
    mse_ae_global = float(ae.reconstruction_mse(X))
except Exception:
    mse_ae_global = None

n_pcs_95_current = int(n_pcs_95) if "n_pcs_95" in globals() else None
mse_pca2 = None
mse_pca95 = None
if "pca" in globals():
    try:
        mse_pca2 = float(pca.reconstruction_error(X, n_components=2).mean())
        if n_pcs_95_current is None:
            n_pcs_95_current = int(pca.n_components_for_variance(0.95))
        mse_pca95 = float(pca.reconstruction_error(X, n_components=n_pcs_95_current).mean())
    except Exception:
        pass

phys_cols = ["teff_gspphot", "logg_gspphot", "mh_gspphot", "bp_rp"]
ae_correlations = {}
for col in phys_cols:
    if col not in meta.columns:
        continue
    vals = pd.to_numeric(meta[col], errors="coerce").to_numpy(dtype=float)
    valid = np.isfinite(vals)
    if valid.sum() < 3:
        ae_correlations[col] = {"latent_axis_1": None, "latent_axis_2": None}
        continue
    r1 = pd.Series(Z_ae_current[valid, 0]).corr(pd.Series(vals[valid]), method="spearman")
    r2 = pd.Series(Z_ae_current[valid, 1]).corr(pd.Series(vals[valid]), method="spearman")
    ae_correlations[col] = {
        "latent_axis_1": float(r1) if pd.notna(r1) else None,
        "latent_axis_2": float(r2) if pd.notna(r2) else None,
    }

latent_distance = None
if "Z_interp" in globals():
    zi = np.asarray(Z_interp)
    if zi.ndim == 2 and zi.shape[0] >= 2:
        latent_distance = float(np.linalg.norm(zi[-1] - zi[0]))

autoencoder_output = {
    "Z_ae": Z_ae_current,
    "X_recon": X_recon_current,
    "y": y,
    "meta": meta,
    "feature_names": feature_names,
    "history": history_current,
    "comparison_df": comparison_df if "comparison_df" in globals() else None,
    "reconstruction_summary": summary if "summary" in globals() else None,
    "method_summary": df_summary if "df_summary" in globals() else None,
    "scores_pca": scores_pca if "scores_pca" in globals() else None,
    "Z_umap": Z_umap if "Z_umap" in globals() else None,
    "Z_tsne": Z_tsne if "Z_tsne" in globals() else None,
    "Z_interp": Z_interp if "Z_interp" in globals() else None,
    "X_interp": X_interp if "X_interp" in globals() else None,
    "idx_cold": int(idx_cold) if "idx_cold" in globals() else None,
    "idx_hot": int(idx_hot) if "idx_hot" in globals() else None,
    "teff_cold": float(teff_cold) if "teff_cold" in globals() else None,
    "teff_hot": float(teff_hot) if "teff_hot" in globals() else None,
    "n_pcs_95": n_pcs_95_current,
    "ae_fit_time_s": float(ae_fit_time) if ae_fit_time is not None else None,
    "ae_mse_global": mse_ae_global,
}

save_path = Path(paths["REPORTS_DIR"]) / "phy3500_autoencoder_output.joblib"
joblib.dump(autoencoder_output, save_path)

report = {
    "run_timestamp_utc": timestamp,
    "paths": {
        "features_path": str(FEATURES_PATH) if "FEATURES_PATH" in globals() else None,
        "catalog_path": str(CATALOG_PATH) if "CATALOG_PATH" in globals() else None,
        "figures_dir": str(FIGURES_DIR),
        "models_dir": str(MODELS_DIR) if "MODELS_DIR" in globals() else None,
        "ae_model_path": str(ae_model_path) if "ae_model_path" in globals() else None,
        "autoencoder_joblib_path": str(save_path),
        "run_dir": str(run_dir),
    },
    "shapes": {
        "X": list(np.asarray(X).shape),
        "Z_ae": list(np.asarray(Z_ae_current).shape),
        "X_recon": list(np.asarray(X_recon_current).shape) if X_recon_current is not None else None,
        "scores_pca": list(np.asarray(scores_pca).shape) if "scores_pca" in globals() else None,
        "meta": list(meta.shape),
    },
    "class_counts": _class_counts(y),
    "metrics": {
        "ae_fit_time_s": float(ae_fit_time) if ae_fit_time is not None else None,
        "ae_epochs_done": int(epochs_done) if epochs_done is not None else None,
        "ae_best_val_loss": best_val_loss,
        "ae_final_train_loss": final_train_loss,
        "ae_final_val_loss": final_val_loss,
        "ae_mse_global": mse_ae_global,
        "pca_mse_2_components": mse_pca2,
        "pca_mse_n_components_95": mse_pca95,
        "n_pcs_95": n_pcs_95_current,
    },
    "correlations": {
        "ae_latent_vs_physical_spearman": ae_correlations,
        "pca_corr_table": _safe_df_records(corr_pca.reset_index()) if "corr_pca" in globals() else None,
    },
    "tables": {
        "comparison_df": _safe_df_records(comparison_df) if "comparison_df" in globals() else None,
        "reconstruction_summary": _safe_df_records(summary) if "summary" in globals() else None,
        "method_summary": (
            _safe_df_records(df_summary.reset_index().rename(columns={"index": "method"}))
            if "df_summary" in globals()
            else None
        ),
    },
    "interpolation": {
        "idx_cold": int(idx_cold) if "idx_cold" in globals() else None,
        "idx_hot": int(idx_hot) if "idx_hot" in globals() else None,
        "teff_cold": float(teff_cold) if "teff_cold" in globals() else None,
        "teff_hot": float(teff_hot) if "teff_hot" in globals() else None,
        "latent_distance": latent_distance,
    },
    "embeddings": {
        "umap_available": bool("Z_umap" in globals() and Z_umap is not None),
        "tsne_available": bool("Z_tsne" in globals() and Z_tsne is not None),
        "umap_stats": _embedding_stats(Z_umap) if "Z_umap" in globals() else None,
        "tsne_stats": _embedding_stats(Z_tsne) if "Z_tsne" in globals() else None,
        "ae_stats": _embedding_stats(Z_ae_current),
    },
    "figures_saved": sorted(str(p) for p in FIGURES_DIR.glob("*.png")),
}

json_path = run_dir / f"phy3500_autoencoder_run_{timestamp}.json"
json_latest_path = run_dir / "phy3500_autoencoder_run_latest.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=_json_default)

with open(json_latest_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=_json_default)

text_lines = [
    "AstroSpectro | PHY-3500 Autoencoder Run Report",
    "=" * 72,
    f"Timestamp (UTC)             : {timestamp}",
    f"Features file               : {FEATURES_PATH if 'FEATURES_PATH' in globals() else 'N/A'}",
    f"Catalog file                : {CATALOG_PATH if 'CATALOG_PATH' in globals() else 'N/A'}",
    f"Figures directory           : {FIGURES_DIR}",
    "",
    "[Saved artifacts]",
    f"- Autoencoder joblib        : {save_path}",
    f"- JSON run report           : {json_path}",
    "",
    "[Shapes]",
    f"- X                         : {np.asarray(X).shape}",
    f"- Z_ae                      : {np.asarray(Z_ae_current).shape}",
    f"- X_recon                   : {np.asarray(X_recon_current).shape if X_recon_current is not None else 'N/A'}",
    "",
    "[Class counts]",
    f"- y                         : {_class_counts(y)}",
    "",
    "[Metrics]",
    f"- AE fit time               : {_fmt_seconds(ae_fit_time)}",
    f"- AE epochs                 : {epochs_done if epochs_done is not None else 'N/A'}",
    f"- AE best val_loss          : {best_val_loss if best_val_loss is not None else 'N/A'}",
    f"- AE final train_loss       : {final_train_loss if final_train_loss is not None else 'N/A'}",
    f"- AE final val_loss         : {final_val_loss if final_val_loss is not None else 'N/A'}",
    f"- AE MSE global             : {mse_ae_global if mse_ae_global is not None else 'N/A'}",
    f"- PCA MSE (2)               : {mse_pca2 if mse_pca2 is not None else 'N/A'}",
    f"- PCA MSE (n95)             : {mse_pca95 if mse_pca95 is not None else 'N/A'}",
    f"- n_pcs_95                  : {n_pcs_95_current if n_pcs_95_current is not None else 'N/A'}",
    "",
    "[AE latent correlations - Spearman]",
    str(ae_correlations),
    "",
]

if "comparison_df" in globals():
    text_lines.extend([
        "Comparison AE vs PCA:",
        comparison_df.round(6).to_string(index=False),
        "",
    ])

if "summary" in globals():
    text_lines.extend([
        "Reconstruction summary by class:",
        summary.round(6).to_string(index=False),
        "",
    ])

if "df_summary" in globals():
    text_lines.extend([
        "Method summary table:",
        df_summary.to_string(),
        "",
    ])

if latent_distance is not None:
    text_lines.extend([
        "Interpolation:",
        f"- idx_cold                  : {int(idx_cold) if 'idx_cold' in globals() else 'N/A'}",
        f"- idx_hot                   : {int(idx_hot) if 'idx_hot' in globals() else 'N/A'}",
        f"- latent_distance           : {latent_distance:.6f}",
        "",
    ])

text_path = run_dir / f"phy3500_autoencoder_run_{timestamp}.txt"
text_latest_path = run_dir / "phy3500_autoencoder_run_latest.txt"
text_content = "\n".join(text_lines)

with open(text_path, "w", encoding="utf-8") as f:
    f.write(text_content)

with open(text_latest_path, "w", encoding="utf-8") as f:
    f.write(text_content)

print(f"Autoencoder output sauvegarde -> {save_path}")
print(f"Rapport JSON run              -> {json_path}")
print(f"Rapport TXT run               -> {text_path}")
print(f"Raccourci JSON latest         -> {json_latest_path}")
print(f"Raccourci TXT latest          -> {text_latest_path}")
print(f"  Z_ae shape                  : {np.asarray(Z_ae_current).shape}")

<a id="conclusion"></a>

<div style="
    background: linear-gradient(135deg, #2D1E2F 0%, #5A2A5E 50%, #2C6E8F 100%);
    padding: 18px 22px;
    border-radius: 12px;
    margin: 18px 0 12px 0;
    box-shadow: 0 8px 24px rgba(45,30,47,0.30);
">
  <h2 style="color: white; margin: 0; font-weight: 350; letter-spacing: 1px;">Résumé final (run 20260405T220839Z)</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 10px;
    padding: 12px 14px;
    margin-bottom: 12px;
">
Tableau complété à partir de <code>data/reports/runs/phy3500_autoencoder/phy3500_autoencoder_run_20260405T220839Z.json</code>.
</div>

| Analyse | Résultat clé (run 20260405T220839Z) |
|---------|--------------------------------------|
| Échantillon | 43 019 objets, 183 features standardisées |
| Entraînement AE | 143.67 s, 145 epochs, best val_loss = 0.54808 |
| Reconstruction AE (z=2) | MSE globale = 0.48910 |
| Référence PCA (2 composantes) | MSE = 0.69627 |
| Référence PCA (n=51, 95% var) | MSE = 0.19624 |
| Corrélation AE axe 2 - T_eff | rho = +0.793 (gradient thermique fort) |
| Corrélation AE axe 2 - BP-RP | rho = -0.770 (cohérence couleur-température) |
| Interpolation latente froide-chaude | distance = 11.443 ; 4199.6 K -> 7984.0 K |

### Lecture scientifique
- À 2 dimensions, l’autoencodeur non linéaire reconstruit mieux que PCA-2 (0.489 vs 0.696), ce qui confirme l’intérêt d’un modèle non linéaire sous forte contrainte dimensionnelle.
- PCA avec davantage de composantes reste naturellement meilleur en erreur pure (MSE 0.196 à 51 composantes), ce qui est attendu car la capacité de reconstruction augmente avec la dimension.
- Dans ce run, le gradient thermique est surtout porté par l’axe latent 2 (rho +0.793 avec T_eff), ce qui illustre la liberté de rotation des axes latents en autoencodeur.
- Les classes minoritaires (GALAXY, QSO) présentent des erreurs bien plus élevées que STAR, en partie à cause du déséquilibre de classes.

### Recommandation pipeline
Pour la modélisation supervisée principale, les features physiques (ou PCA interprétable) restent la base la plus robuste. L’autoencodeur est particulièrement pertinent pour l’exploration non supervisée, la continuité latente et la détection d’objets atypiques.

<div style="text-align: right; margin-top: 10px;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

<div style="
    background: linear-gradient(135deg, #5A2A5E 0%, #2C6E8F 100%);
    padding: 16px 20px;
    border-radius: 10px;
    margin: 18px 0 10px 0;
    box-shadow: 0 6px 18px rgba(90,42,94,0.25);
">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 0.8px;">Références scientifiques et techniques</h2>
</div>

### Fondements autoencodeur et apprentissage profond
1. Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). *Learning representations by back-propagating errors*. Nature, 323, 533-536. (article fondateur de la rétropropagation).
2. Hinton, G. E., & Salakhutdinov, R. R. (2006). *Reducing the dimensionality of data with neural networks*. Science, 313(5786), 504-507. (autoencodeurs pour réduction de dimension).
3. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. (théorie générale des réseaux profonds, régularisation, optimisation).

### Réduction de dimension de référence
4. Jolliffe, I. T., & Cadima, J. (2016). *Principal component analysis: a review and recent developments*. Phil. Trans. A, 374:20150202. (cadre théorique PCA).
5. McInnes, L., Healy, J., & Melville, J. (2018). *UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction*. arXiv:1802.03426. (comparaison conceptuelle avec embeddings non linéaires).

### Corrélation de rang et interprétation physique
6. Spearman, C. (1904). *The proof and measurement of association between two things*. American Journal of Psychology, 15(1), 72-101. (définition de rho de Spearman).

### Contexte astrophysique des données
7. Gaia Collaboration (2023). *Gaia Data Release 3: Summary of the content and survey properties*. Astronomy & Astrophysics, 674, A1.
8. Luo, A.-L., et al. (2019). *The LAMOST DR5 release*. Research in Astronomy and Astrophysics.

### Implémentation Python
9. PyTorch developers. *PyTorch documentation*. https://pytorch.org/docs/stable/
10. scikit-learn developers. *PCA documentation*. https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin-top: 12px;
">
Ces références couvrent les briques mobilisées dans ce notebook : apprentissage latent non linéaire, comparaison avec méthodes de réduction de dimension, validation statistique et contexte astrophysique.
</div>

## Conclusions

### Ce que chaque méthode apporte

| Critère | PCA | UMAP | Autoencodeur |
|---------|-----|------|--------------|
| Linéarité | Linéaire | Non-linéaire | Non-linéaire |
| Paramétrique (transform nouveaux points) | ✓ | ✓ (partiel) | ✓ |
| Erreur de reconstruction (2 dims) | *voir tableau* | N/A | *voir tableau* |
| Interprétabilité des axes | ✓✓ (loadings physiques) | ✗ | ✗ |
| Temps d'exécution | < 1 s | *voir §2* | *voir §2* |
| Stabilité | ✓✓ (déterministe) | *voir stabilité* | Variable |

### Observations physiques

- **PC1** corrèle fortement avec T_eff (température) : la PCA retrouve
  la séquence principale de Hertzsprung-Russell sans supervision.
- **UMAP** révèle des sous-structures discrètes (clusters) invisibles
  en PCA, témoignant de non-linéarités dans l'espace spectral.
- **L'autoencodeur** compresse vers un espace latent continu (validé
  par l'interpolation §5) mais ses axes n'ont pas d'interprétation
  physique directe contrairement aux loadings de la PCA.

### Recommandation pour le pipeline AstroSpectro principal

Pour la **classification XGBoost**, les features physiques directes (V2)
restent préférables : plus interprétables, déterministes, et prouvées
efficaces à 84 %+. La réduction de dimension est plus pertinente pour
l'**exploration non supervisée** et la **détection d'anomalies**.

### Lien avec PHY-3500

Ce notebook illustre le spectre des méthodes numériques enseignées :
- PCA → décomposition spectrale (valeurs propres, SVD, NR chap. 11)
- UMAP → graphes de voisinage, optimisation stochastique
- Autoencodeur → rétropropagation, descente de gradient, régularisation